<a href="https://colab.research.google.com/github/kokololo1599/inf8225_tp/blob/main/Another_copy_of_INF8225_TP3_2026_v10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INF8225 2026, TP3: RNNs, GRUs and Transformers for Machine translation

The goal of this TP is to compare the performance of three different architectures for neural machine translation:
* A vanilla RNN
* A GRU-RNN
* A transformer

You are provided with the code to load and build the pytorch dataset,
and the code for the training loop.
You "only" have to code the GRU and Transformer architectures, the RNN is provided and the notebook should run using the RNN codepath so you can immediately see the model train by running this notebook. There is code to reduce the dataset by some percentage, so the default is to train with only 5% of the data. This leads to fast training but lower performing models.
The use of built-in torch layers such as `nn.GRU`, `nn.RNN` or `nn.Transformer`
is forbidden, as the goal of this TP is to implement the mathematics of the underlying models. You will be given many hints on how to implement the models. Each section that you need to implement is flagged with a TODO statement.

The source sentences in the dataset are in english and the target language is french.

Do not forget to **select the runtime type as GPU!**, and if you want a better GPU you could purchase a small amount of compute to potentially get an H100 or A100 GPU.

**Sources**

* Dataset: [Tab-delimited Bilingual Sentence Pairs](http://www.manythings.org/anki/)

<!---
M. Cettolo, C. Girardi, and M. Federico. 2012. WIT3: Web Inventory of Transcribed and Translated Talks. In Proc. of EAMT, pp. 261-268, Trento, Italy. pdf, bib. [paper](https://aclanthology.org/2012.eamt-1.60.pdf). [website](https://wit3.fbk.eu/2016-01).
-->

* The code is inspired by this [pytorch tutorial](https://pytorch.org/tutorials/beginner/torchtext_translation_tutorial.html).

*This notebook is quite big, use the table of contents to easily navigate through it.*

# Grading:

# Implementations (50 points total)

10 Points for your implementation of the GRU

40 Points for your implementaiton of the Transformer components

# Questions (12 points, 1 point each)
1. Explain the differences between Vanilla RNN, GRU-RNN, Encoder-Decoder Transformer and a Decoder-Only Transformer (which you do not need to implement).

The Vanilla RNN processes sequence token by token sequentially, maintaining a hidden h_i which updated at every step. This process suffers from the Vanishing and Exploding Gradient which makes deep learning difficult.

The GRU-RNN is a gated version of RNN which solves the Vanishing Gradient problem. It uses update and reset gates to control the flow of information which keeps useful information longer and forget information deemed irrelevant, improving the ability to capture long-term dependencies compared to a vanilla RNN.

The Encoder-Decoder transformer uses attention mechanisms instead of recurrence.The encoder processes the source sentence and creates contextual representations while the decoder generates the output sentence using self-attention and encoder-decoder attention. Unlike RNNs, Transformers can process tokens in parallel during training, which significantly improves computational efficiency.

The Decoder-Only Transformer consists only of the decoder stack. It uses causal self-attention, which ensures that each token can only attend to previous tokens in the sequence and not future ones. This architecture is commonly used for autoregressive language generation, where the model predicts the next token based on the tokens generated so far.

2. Why is positionnal encoding necessary in Transformers and not in RNNs?

Transformers process the tokens in parallel, meaning that the model has no inherent knowledge of the token order. This is why transformers need positional encoding, which provides information about the position of each token in the sequence. In contrast, RNNs process sequences step by step, so the order is naturally captured through the hidden state updates over time.

3. Describe the preprocessing process. Detail how the initial dataset is processed before being fed to the translation models.

Before training the translation model, the dataset undergoes several preprocessing steps:

3.1. Text cleaning : Remove special characters or normalize text and onvert text to lowercase if necessary.

3.2. Tokenization: Sentences are split into tokens (words or subwords)

3.3. Vocabulary creation: A mapping between tokens and indices is created.

3.4. Special tokens added
  - SOS (start of sentence)
  - EOS (end of sentence)
  - PAD (padding)

3.5. Sequence padding: Sentences are padded to the same length within a batch.

3.6. Conversion to tensors: Tokens are converted to integer indices so they can be used by neural networks.

4. What is teacher forcing, and how is it used in Transformer training? How does the decoder input differ?

Teacher forcing is a training strategy where the true target token is fed to the decoder as the next input instead of the model's prediction.

5. How are the two types of mask important to the attention mechanism (causal and padding) and how do they work? How do they differ between the encoder and decoder?

Two masks are used in attention mechanisms:

Padding mask: Prevents attention from focusing on padding tokens.Applied in encoder and decoder attention.

Causal mask: Prevents a token from attending to future tokens.Ensures autoregressive behavior.

Encoder: Uses only padding mask.

Decoder: Uses causal mask + padding mask.

6. What is a causal mask, and why is it only used in the decoder?

A causal mask blocks attention to future tokens in the sequence. This ensures the model predicts token t using only tokens before or at position t.
It is only used in the decoder because:

- The decoder generates text sequentially

- Future tokens should not be visible during prediction

The encoder processes the entire input sentence at once, so no causal restriction is needed.

7. Why does the decoder use both self-attention and encoder-decoder attention?
The decoder needs two sources of information which is self-attention and encoder-decoder attention. Self-attention allows the decoder to understand relationships within the generated output sequence. This allows the decoder to combine the target context and the source context.

8. Why is the Transformer model parallelizable, and how does this improve efficiency compared to RNNs?

Transformers do not rely on sequential hidden state updates like RNNs. That means all tokens are processed simultaneously with attention operations computed with matrix multiplication. This allows for GPU parallelization and faster training.

RNNs must compute token t before t+1, which prevent parallelization.

9. How does multi-head self-attention allow the model to capture different aspects of a sentence?

Multi-head attention uses multiple attention heads, each with its own learned projections. Each head focuses on different relationships (ex: synthatic, subject-verb, long-distance, positional dependencies). The output of every head are combined, allowing the model to capture multiple patterns simultaneously.

10. What does the decoder's final output represent before the projection layer? What does the encoder's final output represent?

The encoder's final output is a sequence of contextual embeddings, where each vector represents a token in the input sentence enriched with information from all other tokens in the sequence.

The decoder's final output before the projection layer is also a sequence of contextual representations. Each vector represents a generated token and contains information from previous generated tokens and the encoder's representations.

11. What is the role of the final linear projection layer in the decoder?
How does the decoder output differ between training (parallel processing) and inference (sequential generation)?

The final linear layer maps the decoder hidden representation to the vocabulary size. hidden_dim → vocab_size

This produces logits, which represent the probability of each word in the vocabulary being the next token. After this, a softmax converts logits into probabilities.

#### Decoder output during training vs inference

Training

- Entire sequence processed in parallel using teacher forcing.

- Ground truth tokens are used as decoder inputs.

Inference

- Tokens are generated one by one.

- Each predicted token becomes the next decoder input.


12. Why does the decoder recompute all outputs at each inference step instead of appending new outputs incrementally?

At each inference step, the attention mechanism must compute relationships between the current token and all previously generated tokens.

Because attention depends on the full sequence of previous tokens, the decoder recomputes the outputs to correctly update the attention weights.

In optimized implementations, key and value states can be cached, but conceptually the model recomputes the sequence to ensure correct attention calculations.

# You Must Create a Small Report - performing experiments of your own choice, some basic experiments and exploring something interesting of your own choice (15 points for basic experiments + 10 points for exploring your own ideas)
Once everything is working fine, you need to explore aspects of these models and do some research of your own into how they behave. Your report can be submitted as a fully run notebook rendered as a .pdf, or as a document in .pdf format.

For example, how do an RNN, GRU and Transformer compare in terms of performance. How does the model size influence training vs test set performance. How important are hyperparameter settings. What are the effect of the differents hyperparameters with the final model performance? What about training time? How does greedy search compare to beam search ?

What are some other metrics you could have for machine translation? Can you compute them and add them to your WandB report?

Those are only examples, you can do whatever you think will be interesting.
This part accounts for many points, *feel free to go wild!* You may use coding agents aggressively for this part of the TP. This might allow you to integrate much larger ideas into this codebase. For example you may be able examine some of the ideas implemented in this other codebase
https://github.com/jammastergirish/BuildAnLLM
and implment them in this TPs framework.

---

# Resources:

**A great tutorial on pytorch can be found [here](https://stanford.edu/class/cs224n/materials/CS224N_PyTorch_Tutorial.html).**
Spending 3 hours on this tutorial is *no* waste of time.

Several resources could be particularly helpful, including:
Lynn-Evans, Samuel. (2018). How to code The Transformer in Pytorch.
https://towardsdatascience.com/how-to-code-the-transformer-in-pytorch-24db27c8f9ec

Lippe, Phillipe. (2022). Tutorial 6: Transformers and Multi-Head Attention.
https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial6/Transformers_and_MHAttention.html

An excellent larger scale implementation of different types of Transformers
https://github.com/jammastergirish/BuildAnLLM

You can find a basic presentation of `einops` [here](https://einops.rocks/1-einops-basics/).

Here is a formal academic paper about einops [here](https://openreview.net/pdf?id=oapKSVM2bcj)



# Imports


In [ ]:
!python3 -m spacy download en_core_web_sm > /dev/null
!python3 -m spacy download fr_core_news_sm > /dev/null
!pip install torchinfo > /dev/null
!pip install einops > /dev/null
!pip install wandb > /dev/null


from itertools import takewhile
from collections import Counter, defaultdict

import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

import spacy

import einops
import wandb
from torchinfo import summary

In [ ]:
!nvidia-smi -L

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Data and tokenization

We first download and parse the dataset. From the parsed sentences
we can build the vocabularies and the torch datasets.
The end goal of this section is to have an iterator
that can yield the pairs of translated datasets, and
where each sentences is made of a sequence of tokens.

The tokenizers are objects that are able to divide a python string into a list of tokens (words, punctuations, special tokens...) as a list of strings.

The special tokens are used for a particular reasons:
* *\<unk\>*: Replace an unknown word in the vocabulary by this default token
* *\<pad\>*: Virtual token used to as padding token so a batch of sentences can have a unique length
* *\<bos\>*: Token indicating the beggining of a sentence in the target sequence
* *\<eos\>*: Token indicating the end of a sentence in the target sequence

In [ ]:

# Our current dataset is
!wget http://www.manythings.org/anki/fra-eng.zip
!unzip fra-eng.zip


df = pd.read_csv('fra.txt', sep='\t', names=['english', 'french', 'attribution'])
train = [
    (en, fr) for en, fr in zip(df['english'], df['french'])
]
train, valid = train_test_split(train, test_size=0.1, random_state=0)
print(len(train))

# Optional: Reduce dataset size for faster training
USE_REDUCED_DATASET = True  # Set to True to use smaller dataset
DATASET_FRACTION = 0.50  # Use 50% of data

if USE_REDUCED_DATASET:
    train = train[:int(len(train) * DATASET_FRACTION)]
    print(f"Using reduced training set: {len(train)} examples ({DATASET_FRACTION*100}%)")
else:
    print(f"Using full training set: {len(train)} examples")

print(len(train))

# Load spacy models
en_nlp = spacy.load('en_core_web_sm')
fr_nlp = spacy.load('fr_core_news_sm')

# Create tokenizer functions
def en_tokenizer(text):
    return [token.text for token in en_nlp.tokenizer(text)]

def fr_tokenizer(text):
    return [token.text for token in fr_nlp.tokenizer(text)]

SPECIALS = ['<unk>', '<pad>', '<bos>', '<eos>']


# An older dataset, (there's an issue on Colab with it, so we will not use it)
# train, valid, _ = IWSLT2016(language_pair=('fr', 'en'))
# train, valid = list(train), list(valid)

# Another dataset, but it is too large
# !wget https://www.statmt.org/wmt14/training-monolingual-europarl-v7/europarl-v7.en.gz
# !wget https://www.statmt.org/wmt14/training-monolingual-europarl-v7/europarl-v7.fr.gz
# !gunzip europarl-v7.en.gz
# !gunzip europarl-v7.fr.gz

# with open('europarl-v7.en', 'r') as my_file:
#     english = my_file.readlines()

# with open('europarl-v7.fr', 'r') as my_file:
#     french = my_file.readlines()

# dataset = [
#     (en, fr)
#     for en, fr in zip(english, french)
# ]
# print(f'\n{len(dataset):,} sentences.')

# dataset, _ = train_test_split(dataset, test_size=0.8, random_state=0)  # Remove 80% of the dataset (it would be huge otherwise)
# train, valid = train_test_split(dataset, test_size=0.2, random_state=0)  # Split between train and validation dataset


--2026-03-22 17:08:20--  http://www.manythings.org/anki/fra-eng.zip
Resolving www.manythings.org (www.manythings.org)... 173.254.30.110, 173.254.30.110
Connecting to www.manythings.org (www.manythings.org)|173.254.30.110|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8242175 (7.9M) [application/zip]
Saving to: ‘fra-eng.zip’

fra-eng.zip         100%[===================>]   7.86M  6.68MB/s    in 1.2s    

2026-03-22 17:08:21 (6.68 MB/s) - ‘fra-eng.zip’ saved [8242175/8242175]

Archive:  fra-eng.zip
  inflating: _about.txt              
  inflating: fra.txt                 
216468
Using reduced training set: 108234 examples (50.0%)
108234


## Replacement for torchtext tokenization

Functions and classes to build the vocabularies and the torch datasets.
The vocabulary is an object able to transform a string token into the id (an int) of that token in the vocabulary.

In [ ]:
# Custom Vocabulary class (replaces torchtext.vocab.Vocab)
class Vocab:
    """Simple vocabulary class for token-to-index mapping."""

    def __init__(self, token_to_idx, default_index=None):
        self.token_to_idx = token_to_idx
        self.idx_to_token = {idx: token for token, idx in token_to_idx.items()}
        self.default_index = default_index

    def __len__(self):
        return len(self.token_to_idx)

    def __getitem__(self, token):
        """Get index of a token."""
        if self.default_index is not None:
            return self.token_to_idx.get(token, self.default_index)
        return self.token_to_idx[token]

    def __call__(self, tokens):
        """Convert list of tokens to list of indices."""
        return [self[token] for token in tokens]

    def set_default_index(self, default_index):
        """Set the default index for unknown tokens."""
        self.default_index = default_index

    def lookup_token(self, idx):
        """Get token from index."""
        return self.idx_to_token.get(idx, '<unk>')

    def lookup_tokens(self, indices):
        """Get list of tokens from list of indices."""
        return [self.lookup_token(idx) for idx in indices]


def build_vocab_from_iterator(iterator, min_freq=1, specials=None):
    """Build vocabulary from an iterator of token lists.

    Args:
        iterator: Iterator yielding lists of tokens
        min_freq: Minimum frequency for a token to be included
        specials: List of special tokens to add at the beginning

    Returns:
        Vocab object
    """
    # Count token frequencies
    counter = Counter()
    for tokens in iterator:
        counter.update(tokens)

    # Build token to index mapping
    token_to_idx = {}

    # Add special tokens first
    if specials:
        for token in specials:
            token_to_idx[token] = len(token_to_idx)

    # Add tokens that meet minimum frequency
    for token, freq in counter.items():
        if freq >= min_freq and token not in token_to_idx:
            token_to_idx[token] = len(token_to_idx)

    return Vocab(token_to_idx)

In [ ]:
class TranslationDataset(Dataset):
    def __init__(
            self,
            dataset: list,
            en_vocab: Vocab,
            fr_vocab: Vocab,
            en_tokenizer,
            fr_tokenizer,
        ):
        super().__init__()

        self.dataset = dataset
        self.en_vocab = en_vocab
        self.fr_vocab = fr_vocab
        self.en_tokenizer = en_tokenizer
        self.fr_tokenizer = fr_tokenizer

    def __len__(self):
        """Return the number of examples in the dataset.
        """
        return len(self.dataset)

    def __getitem__(self, index: int) -> tuple:
        """Return a sample.

        Args
        ----
            index: Index of the sample.

        Output
        ------
            en_tokens: English tokens of the sample, as a LongTensor.
            fr_tokens: French tokens of the sample, as a LongTensor.
        """
        # Get the strings
        en_sentence, fr_sentence = self.dataset[index]

        # To list of words
        # We also add the beggining-of-sentence and end-of-sentence tokens
        en_tokens = ['<bos>'] + self.en_tokenizer(en_sentence) + ['<eos>']
        fr_tokens = ['<bos>'] + self.fr_tokenizer(fr_sentence) + ['<eos>']

        # To list of tokens
        en_tokens = self.en_vocab(en_tokens)  # list[int]
        fr_tokens = self.fr_vocab(fr_tokens)

        return torch.LongTensor(en_tokens), torch.LongTensor(fr_tokens)


def yield_tokens(dataset, tokenizer, lang):
    """Tokenize the whole dataset and yield the tokens.
    """
    assert lang in ('en', 'fr')
    sentence_idx = 0 if lang == 'en' else 1

    for sentences in dataset:
        sentence = sentences[sentence_idx]
        tokens = tokenizer(sentence)
        yield tokens


def build_vocab(dataset: list, en_tokenizer, fr_tokenizer, min_freq: int):
    """Return two vocabularies, one for each language.
    """
    en_vocab = build_vocab_from_iterator(
        yield_tokens(dataset, en_tokenizer, 'en'),
        min_freq=min_freq,
        specials=SPECIALS,
    )
    en_vocab.set_default_index(en_vocab['<unk>'])  # Default token for unknown words

    fr_vocab = build_vocab_from_iterator(
        yield_tokens(dataset, fr_tokenizer, 'fr'),
        min_freq=min_freq,
        specials=SPECIALS,
    )
    fr_vocab.set_default_index(fr_vocab['<unk>'])

    return en_vocab, fr_vocab


def preprocess(
        dataset: list,
        en_tokenizer,
        fr_tokenizer,
        max_words: int,
    ) -> list:
    """Preprocess the dataset.
    Remove samples where at least one of the sentences are too long.
    Those samples takes too much memory.
    Also remove the pending '\n' at the end of sentences.
    """
    filtered = []

    for en_s, fr_s in dataset:
        if len(en_tokenizer(en_s)) >= max_words or len(fr_tokenizer(fr_s)) >= max_words:
            continue

        en_s = en_s.replace('\n', '')
        fr_s = fr_s.replace('\n', '')

        filtered.append((en_s, fr_s))

    return filtered


def build_datasets(
        max_sequence_length: int,
        min_token_freq: int,
        en_tokenizer,
        fr_tokenizer,
        train: list,
        val: list,
    ) -> tuple:
    """Build the training, validation and testing datasets.
    It takes care of the vocabulary creation.

    Args
    ----
        - max_sequence_length: Maximum number of tokens in each sequences.
            Having big sequences increases dramatically the VRAM taken during training.
        - min_token_freq: Minimum number of occurences each token must have
            to be saved in the vocabulary. Reducing this number increases
            the vocabularies's size.
        - en_tokenizer: Tokenizer for the english sentences.
        - fr_tokenizer: Tokenizer for the french sentences.
        - train and val: List containing the pairs (english, french) sentences.


    Output
    ------
        - (train_dataset, val_dataset): Tuple of the two TranslationDataset objects.
    """
    datasets = [
        preprocess(samples, en_tokenizer, fr_tokenizer, max_sequence_length)
        for samples in [train, val]
    ]

    en_vocab, fr_vocab = build_vocab(datasets[0], en_tokenizer, fr_tokenizer, min_token_freq)

    datasets = [
        TranslationDataset(samples, en_vocab, fr_vocab, en_tokenizer, fr_tokenizer)
        for samples in datasets
    ]

    return datasets

In [ ]:
def generate_batch(data_batch: list, src_pad_idx: int, tgt_pad_idx: int) -> tuple:
    """Add padding to the given batch so that all
    the samples are of the same size.

    Args
    ----
        data_batch: List of samples.
            Each sample is a tuple of LongTensors of varying size.
        src_pad_idx: Source padding index value.
        tgt_pad_idx: Target padding index value.

    Output
    ------
        en_batch: Batch of tokens for the padded english sentences.
            Shape of [batch_size, max_en_len].
        fr_batch: Batch of tokens for the padded french sentences.
            Shape of [batch_size, max_fr_len].
    """
    en_batch, fr_batch = [], []
    for en_tokens, fr_tokens in data_batch:
        en_batch.append(en_tokens)
        fr_batch.append(fr_tokens)

    en_batch = pad_sequence(en_batch, padding_value=src_pad_idx, batch_first=True)
    fr_batch = pad_sequence(fr_batch, padding_value=tgt_pad_idx, batch_first=True)
    return en_batch, fr_batch

# Models architecture
This is where you have to code the architectures.

In a machine translation task, the model takes as input the whole
source sentence along with the current known tokens of the target,
and predict the next token in the target sequence.
This means that the target tokens are predicted in an autoregressive
manner, starting from the first token (right after the *\<bos\>* token) and producing tokens one by one until the last *\<eos\>* token.

Formally, we define $s = [s_1, ..., s_{N_s}]$ as the source sequence made of $N_s$ tokens.
We also define $t^i = [t_1, ..., t_i]$ as the target sequence at the beginning of the step $i$.

The output of the model parameterized by $\theta$ is:

$$
T_{i+1} = p(t_{i+1} | s, t^i ; \theta )
$$

Where $T_{i+1}$ is the distribution of the next token $t_{i+1}$.

The loss is simply a *cross entropy loss* over the whole steps, where each class is a token of the vocabulary.

![RNN schema for machinea translation](https://www.simplilearn.com/ice9/free_resources_article_thumb/machine-translation-model-with-encoder-decoder-rnn.jpg)

Note that in this image the english sentence is provided in reverse.

---

In pytorch, there is no dinstinction between an intermediate layer or a whole model having multiple layers in itself.
Every layers or models inherit from the `torch.nn.Module`.
This module needs to define the `__init__` method where you instanciate the layers,
and the `forward` method where you decide how the inputs and the layers of the module interact between them.
Thanks to the autograd computations of pytorch, you do not have
to implement any backward method!

A really important advice is to **always look at
the shape of your input and your output.**
From that, you can often guess how the layers should interact
with the inputs to produce the right output.
You can also easily detect if there's something wrong going on.

You are more than advised to use the `einops` library and the `torch.einsum` function. This will require less operations than 'classical' code, but note that it's a bit trickier to use.
This is a way of describing tensors manipulation with strings, bypassing the multiple tensor methods executed in the background.

Reminders:
You can find a basic presentation of `einops` [here](https://einops.rocks/1-einops-basics/).
Here is a formal academic paper about einops [here](https://openreview.net/pdf?id=oapKSVM2bcj)

**A great tutorial on pytorch can be found [here](https://stanford.edu/class/cs224n/materials/CS224N_PyTorch_Tutorial.html).**
Spending 3 hours on this tutorial is *no* waste of time.

## RNN models

### RNN
Here you have to implement a recurrent neural network. You will need to create a single RNN Layer, and a module allowing to stack these layers. Look up the pytorch documentation to figure out this module's operations and what is communicated from one layer to another


In [ ]:
"""
This snippet comes from:
harvardnlp. (3 avril 2018). The Illustrated Transformer.
URL: https://nlp.seas.harvard.edu/2018/04/03/attention.html
"""
import copy
def clones(module, N):
    "Produce N identical layers."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [ ]:
from einops import rearrange
class RNNCell(nn.Module):
    """A single RNN layer.

    Parameters
    ----------
        input_size: Size of each input token.
        hidden_size: Size of each RNN hidden state.
        dropout: Dropout rate.
    """
    def __init__(
            self,
            input_size: int,
            hidden_size: int,
            dropout: float,
        ):
        super().__init__()

        self.linear_1 = nn.Linear(input_size, hidden_size).to(device)
        self.linear_2 = nn.Linear(hidden_size, hidden_size).to(device)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x: torch.FloatTensor, h: torch.FloatTensor) -> tuple:
        """Go through all the sequence in x, iteratively updating
        the hidden state h.

        Args
        ----
            x: Input sequence.
                Shape of [batch_size, seq_len, input_size].
            h: Initial hidden state.
                Shape of [batch_size, hidden_size].

        Output
        ------
            y: Token embeddings.
                Shape of [batch_size, seq_len, hidden_size].
            h: Last hidden state.
                Shape of [batch_size, hidden_size].
        """
        x = x.transpose(0,1)
        sl = x.size(0)
        bs = x.size(1)
        ins = h.size(1)

        y = torch.zeros(sl, bs, ins).to(device)
        for i, s in enumerate(x):
          h = torch.tanh(self.linear_1(s) + self.linear_2(h))
          h = self.dropout(h)
          y[i] = h

        y = y.transpose(0,1)
        return y, h

class RNN(nn.Module):
    """Implementation of an RNN based
    on https://pytorch.org/docs/stable/generated/torch.nn.RNN.html.

    Parameters
    ----------
        input_size: Size of each input token.
        hidden_size: Size of each RNN hidden state.
        num_layers: Number of layers (RNNCell or GRUCell).
        dropout: Dropout rate.
        model_type: Either 'RNN' or 'GRU', to select which model we want.
    """
    def __init__(
            self,
            input_size: int,
            hidden_size: int,
            num_layers: int,
            dropout: float
        ):
      super().__init__()

      self.input_size = input_size
      self.hidden_size = hidden_size
      self.num_layers = num_layers
      self.layers = nn.ModuleList([RNNCell(input_size, hidden_size, dropout)])
      self.layers.extend(clones(RNNCell(hidden_size, hidden_size, dropout), num_layers-1))


    def forward(self, x: torch.FloatTensor, h: torch.FloatTensor=None) -> tuple:
      """Pass the input sequence through all the RNN cells.
      Returns the output and the final hidden state of each RNN layer

      Args
      ----
          x: Input sequence.
              Shape of [batch_size, seq_len, input_size].
          h: Hidden state for each RNN layer.
              Can be None, in which case an initial hidden state is created.
              Shape of [batch_size, n_layers, hidden_size].

      Output
      ------
          y: Output embeddings for each token after the RNN layers.
              Shape of [batch_size, seq_len, hidden_size].
          h: Final hidden state.
              Shape of [batch_size, n_layers, hidden_size].
      """
      if h is None:
        h = torch.zeros(x.size(0), self.num_layers, self.hidden_size).to(device)

      h_next = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
      h = h.transpose(0,1)

      out = x
      for i, layer in enumerate(self.layers):
        out, h_next[i] = layer(out, h[i])

      h_next = h_next.transpose(0,1)
      y = out
      return y, h_next

### GRU
Here you have to implement a GRU-RNN. This architecture is close to the Vanilla RNN but perform different operations. Look up the pytorch documentation to figure out the differences.

In [ ]:

class GRUCell(nn.Module):
    """A single GRU layer.

    Parameters
    ----------
        input_size: Size of each input token.
        hidden_size: Size of each RNN hidden state.
        dropout: Dropout rate.
    """
    def __init__(
            self,
            input_size: int,
            hidden_size: int,
            dropout: float,
        ):
        super().__init__()

        # TODO
        # Initialize six linear transformation layers for the GRU gates:
        # - linear_i_r: transforms input to reset gate (input_size → hidden_size)
        # - linear_h_r: transforms hidden state to reset gate (hidden_size → hidden_size)
        # - linear_i_z: transforms input to update gate (input_size → hidden_size)
        # - linear_h_z: transforms hidden state to update gate (hidden_size → hidden_size)
        # - linear_i_n: transforms input to new gate/candidate (input_size → hidden_size)
        # - linear_h_n: transforms hidden state to new gate (hidden_size → hidden_size)
        # Syntax: self.layer_name = nn.Linear(...).to(device)
        # Then create a dropout layer: self.dropout = nn.Dropout(dropout)
        self.linear_i_r = nn.Linear(input_size, hidden_size).to(device)
        self.linear_h_r = nn.Linear(hidden_size, hidden_size).to(device)
        self.linear_i_z = nn.Linear(input_size, hidden_size).to(device)
        self.linear_h_z = nn.Linear(hidden_size, hidden_size).to(device)
        self.linear_i_n = nn.Linear(input_size, hidden_size).to(device)
        self.linear_h_n = nn.Linear(hidden_size, hidden_size).to(device)

        self.dropout = nn.Dropout(dropout)


    def forward(self, x: torch.FloatTensor, h: torch.FloatTensor) -> tuple:
        """
        Args
        ----
            x: Input sequence.
                Shape of [batch_size, seq_len, input_size].
            h: Initial hidden state.
                Shape of [batch_size, hidden_size].

        Output
        ------
            y: Token embeddings.
                Shape of [batch_size, seq_len, hidden_size].
            h: Last hidden state.
                Shape of [batch_size, hidden_size].
        """

        # TODO
        # Implement the GRU forward pass:
        # 1. Transpose input from [batch, seq_len, input_size] to [seq_len, batch, input_size]
        #    using x = x.transpose(0,1)
        # 2. Extract dimensions: sl = x.size(0), bs = x.size(1), ins = h.size(1)
        # 3. Initialize output tensor: y = torch.zeros(sl, bs, ins).to(device)
        # 4. Loop over sequence: for i, s in enumerate(x):
        # 5. Compute reset gate: r = torch.sigmoid(self.linear_i_r(s) + self.linear_h_r(h))
        # 6. Compute update gate: z = ...
        # 7. Compute candidate: n = ...
        # 8. Update hidden state:
        # 9. Apply dropout: h = ...
        # 10. Store in output: y[i] = h
        # 11. Return transposed y and final h
        x = x.transpose(0,1)
        sl = x.size(0)
        bs = x.size(1)
        ins = h.size(1)
        y = torch.zeros(sl, bs, ins).to(device)

        for i, s in enumerate(x):
          r = torch.sigmoid(self.linear_i_r(s) + self.linear_h_r(h))
          z = torch.sigmoid(self.linear_i_z(s) + self.linear_h_z(h))
          n = torch.tanh(self.linear_i_n(s) + r * self.linear_h_n(h))

          h = (1 - z) * n + z * h
          h = self.dropout(h)

          y[i] = h
        y = y.transpose(0,1)
        return y, h

class GRU(nn.Module):
    """Implementation of a GRU based on https://pytorch.org/docs/stable/generated/torch.nn.GRU.html.

    Parameters
    ----------
        input_size: Size of each input token.
        hidden_size: Size of each RNN hidden state.
        num_layers: Number of layers.
        dropout: Dropout rate.
    """
    def __init__(
            self,
            input_size: int,
            hidden_size: int,
            num_layers: int,
            dropout: float,
        ):

        super().__init__()

        # TODO
        # Store dimensions as instance variables
        # Create a ModuleList to hold all GRU layers:
        # - First layer: GRUCell(input_size, hidden_size, dropout)
        # - Remaining layers: GRUCell(hidden_size, hidden_size, dropout) × (num_layers-1)
        # Syntax: self.layers = nn.ModuleList([GRUCell(...)])
        # Then extend: self.layers.extend(clones(GRUCell(hidden_size, hidden_size, dropout), num_layers-1))

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.layers = nn.ModuleList()

        # First layer
        self.layers.append(GRUCell(input_size, hidden_size, dropout))

        # Remaining layers
        for _ in range(num_layers-1):
          self.layers.append(GRUCell(hidden_size, hidden_size, dropout))

        #self.layers.extend(clones(GRUCell(hidden_size, hidden_size, dropout), num_layers-1))



    def forward(self, x: torch.FloatTensor, h: torch.FloatTensor=None) -> tuple:
        """
        Args
        ----
            x: Input sequence
                Shape of [batch_size, seq_len, input_size].
            h: Initial hidden state for each layer.
                If 'None', then an initial hidden state (a zero filled tensor)
                is created.
                Shape of [batch_size, n_layers, hidden_size].

        Output
        ------
            output:
                Shape of [batch_size, seq_len, hidden_size].
            h_n: Final hidden state.
                Shape of [batch_size, n_layers, hidden size].
        """

        # TODO
        # Implement multi-layer GRU forward pass:
        # 1. Initialize hidden state if None:
        #    if h == None:  h = torch.zeros(x.size(0), self.num_layers, self.hidden_size).to(device)
        #    Shape: [batch_size, num_layers, hidden_size]
        # 2. Create tensor for next hidden states:
        #    h_next = torch.zeros...
        #    Shape: [num_layers, batch_size, hidden_size]
        # 3. Transpose h from [batch, layers, hidden] to [layers, batch, hidden]:
        #
        # 4. Initialize output with input: out = x
        # 5. Loop through each GRU layer sequentially:
        #    for i, layer in enumerate(self.layers):
        #        out, h_next[i] = layer(out, h[i])
        #    Each layer takes output from previous layer and its corresponding hidden state
        # 6. Transpose h_next back to [batch, layers, hidden]:
        #    h_next = h_next.transpose(0,1)
        # 7. Assign final output: y = out
        # 8. Return (final layer output) and (all layers' hidden states)
        if h == None:
          h = torch.zeros(x.size(0), self.num_layers, self.hidden_size).to(device)

        # [batch, layers, hidden] → [layers, batch, hidden]
        h = h.transpose(0, 1)
        h_next = torch.zeros_like(h)


        out = x
        for i, layer in enumerate(self.layers):
          out, h_next[i] = layer(out, h[i])

        # Back to [batch, layers, hidden]
        h_next = h_next.transpose(0, 1)

        return out, h_next


### Translation RNN

This module instanciates a vanilla RNN or a GRU-RNN and performs the translation task. You have to:
* Encode the source sequence
* Pass the final hidden state of the encoder to the decoder(one for each layer)
* Decode the hidden state into the target sequence

We use teacher forcing for training, meaning that when the next token is predicted, that prediction is based on the previous true target tokens.

In [ ]:
class TranslationRNN(nn.Module):
    """Basic RNN encoder and decoder for a translation task.
    It can run as a vanilla RNN or a GRU-RNN.

    Parameters
    ----------
        n_tokens_src: Number of tokens in the source vocabulary.
        n_tokens_tgt: Number of tokens in the target vocabulary.
        dim_embedding: Dimension size of the word embeddings (for both language).
        dim_hidden: Dimension size of the hidden layers in the RNNs
            (for both the encoder and the decoder).
        n_layers: Number of layers in the RNNs.
        dropout: Dropout rate.
        src_pad_idx: Source padding index value.
        tgt_pad_idx: Target padding index value.
        model_type: Either 'RNN' or 'GRU', to select which model we want.
    """

    def __init__(
            self,
            n_tokens_src: int,
            n_tokens_tgt: int,
            dim_embedding: int,
            dim_hidden: int,
            n_layers: int,
            dropout: float,
            src_pad_idx: int,
            tgt_pad_idx: int,
            model_type: str
        ):
        super().__init__()

        self.src_embeddings = nn.Embedding(n_tokens_src, dim_embedding, src_pad_idx)
        self.tgt_embeddings = nn.Embedding(n_tokens_tgt, dim_embedding, tgt_pad_idx)

        self.dropout_1 = nn.Dropout(dropout)
        self.dropout_2 = nn.Dropout(dropout)

        Model = RNN if model_type == "RNN" else GRU

        self.encoder = Model(dim_embedding, dim_hidden, n_layers, dropout)
        self.norm = nn.LayerNorm(dim_hidden)
        self.decoder = Model(dim_embedding, dim_hidden, n_layers, dropout)

        self.seq = nn.Sequential(
            nn.Linear(dim_hidden, dim_hidden),
            nn.LeakyReLU(),
            nn.LayerNorm(dim_hidden),
            nn.Linear(dim_hidden, dim_hidden),
            nn.LeakyReLU(),
            nn.LayerNorm(dim_hidden),
            nn.Linear(dim_hidden, dim_hidden),
            nn.LeakyReLU(),
            nn.LayerNorm(dim_hidden),
            nn.Linear(dim_hidden, n_tokens_tgt)
        )

    def forward(
        self,
        source: torch.LongTensor,
        target: torch.LongTensor
    ) -> torch.FloatTensor:
        """Predict the source tokens based on the target tokens.

        Args
        ----
            source: Batch of source sentences.
                Shape of [batch_size, src_seq_len].
            target: Batch of target sentences.
                Shape of [batch_size, tgt_seq_len].

        Output
        ------
            y: Batch of predictions where each token is the prediction of
                the next token in the sentence.
                Shape of [batch_size, tgt_seq_len].
        """

        # Reversing the source sequences
        source = torch.fliplr(source)

        src_emb = self.src_embeddings(source)
        out, hidden = self.encoder(src_emb)

        hidden = self.norm(hidden)

        tgt_emb = self.tgt_embeddings(target)
        y, hidden = self.decoder(tgt_emb, hidden)

        y = self.seq(y)

        return y



## Transformer model
Here you have to code the Transformer architecture.
It is divided in three parts:
* Attention layers
* Encoder and decoder layers
* Main layers (gather the encoder and decoder layers)

The [illustrated transformer](https://jalammar.github.io/illustrated-transformer/) blog can help you
understanding how the architecture works.
Once this is done, you can use [the annontated transformer](https://nlp.seas.harvard.edu/2018/04/03/attention.html) to have an idea of how to code this architecture.
We encourage you to use `torch.einsum` and the `einops` library as much as you can. It will make your code simpler.

---
**Implementation order**

To help you with the implementation, we advise you following this order:
* Implement `TranslationTransformer` and use `nn.Transformer` instead of `Transformer`
* Implement `Transformer` and use `nn.TransformerDecoder` and `nn.TransformerEnocder`
* Implement the `TransformerDecoder` and `TransformerEncoder` and use `nn.MultiHeadAttention`
* Implement `MultiHeadAttention`

Do not forget to add `batch_first=True` when necessary in the `nn` modules.

### Attention layers
We use a `MultiHeadAttention` module, that is able to perform self-attention aswell as cross-attention (depending on what you give as queries, keys and values).

**Attention**


It takes the multiheaded queries, keys and values as input.
It computes the attention between the queries and the keys and return the attended values.

The implementation of this function can greatly be improved with *einsums*.

**MultiheadAttention**

Computes the multihead queries, keys and values and feed them to the `attention` function.
You also need to merge the key padding mask and the attention mask into one mask.

The implementation of this module can greatly be improved with *einops.rearrange*.

In [ ]:
from einops.layers.torch import Rearrange
from einops import rearrange
import math
import torch.nn.functional as F
import copy

def attention(
        q: torch.FloatTensor,
        k: torch.FloatTensor,
        v: torch.FloatTensor,
        mask: torch.BoolTensor=None,
        dropout: nn.Dropout=None,
    ) -> tuple:
    """Computes multihead scaled dot-product attention from the
    projected queries, keys and values.

    Args
    ----
        q: Batch of queries.
            Shape of [batch_size, seq_len_1, n_heads, dim_model].
        k: Batch of keys.
            Shape of [batch_size, seq_len_2, n_heads, dim_model].
        v: Batch of values.
            Shape of [batch_size, seq_len_2, n_heads, dim_model].
        mask: Prevent tokens to attend to some other tokens (for padding or autoregressive attention).
            Attention is prevented where the mask is `True`.
            Shape of [batch_size, n_heads, seq_len_1, seq_len_2],
            or broadcastable to that shape.
        dropout: Dropout layer to use.

    Output
    ------
        y: Multihead scaled dot-attention between the queries, keys and values.
            Shape of [batch_size, seq_len_1, n_heads, dim_model].
        attn: Computed attention mask.
            Shape of [batch_size, n_heads, seq_len_1, seq_len_2].
    """
    # TODO
    # Implement scaled dot-product attention:
    # 1. Get dimension of each head: dim_model = q.size(-1)
    # 2. Compute attention scores using Einstein summation:
    #    attn = torch.einsum(...) / sqrt(model dimension)
    #    This should compute Q·K^T / sqrt(d_k) where:
    #    - b=batch, h=heads, l=query_seq_len, t=key_seq_len, k=dim_per_head
    # 3. Apply mask if provided: attn = attn.masked_fill(mask==0, -math.inf)
    #    This sets masked positions to -inf so softmax gives them 0 weight
    # 4. Apply softmax: attn = torch.softmax(attn, dim=-1)
    # 5. Apply dropout if provided: attn = dropout(attn)
    # 6. Compute weighted sum of values: y = torch.einsum(..., [attn, v])
    # 7. Return both the output y and attention weights attn

    # See the following references:
    # Lynn-Evans, Samuel. (2018). How to code The Transformer in Pytorch. https://towardsdatascience.com/how-to-code-the-transformer-in-pytorch-24db27c8f9ec
    # Lippe, Phillipe. (2022). Tutorial 6: Transformers and Multi-Head Attention. https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial6/Transformers_and_MHAttention.html
    dim_model = q.size(-1)
    attn = torch.einsum("bhld,bhtd->bhlt", q, k) / math.sqrt(dim_model)
    if mask is not None:
      attn = attn.masked_fill(mask == 0, -math.inf)

    attn = torch.softmax(attn, dim=-1)
    if dropout is not None:
        attn = dropout(attn)
    y = torch.einsum('bhlt,bhtd->bhld', attn, v)
    return y, attn

class MultiheadAttention(nn.Module):
    """Multihead attention module.
    Can be used as a self-attention and cross-attention layer.
    The queries, keys and values are projected into multiple heads
    before computing the attention between those tensors.

    Parameters
    ----------
        dim: Dimension of the input tokens.
        n_heads: Number of heads. `dim` must be divisible by `n_heads`.
        dropout: Dropout rate.
    """
    def __init__(
            self,
            dim: int,
            n_heads: int,
            dropout: float,
        ):
        super().__init__()

        # TODO
        # Store attention parameters (self.):
        # dim = (total embedding dimension)
        # d_k = (dimension per head)
        # n_heads = (number of attention heads)
        # Create three linear projection layers (no bias):
        # - self.q_linear = nn.Linear(dim, dim)  for queries
        # - self.k_linear = ... for keys
        # - self.v_linear = ... for values
        # Create a dropout layer
        # Create output projection: self.out = ...
        # See the following for help:
        # Lynn-Evans, Samuel. (2018). How to code The Transformer in Pytorch. https://towardsdatascience.com/how-to-code-the-transformer-in-pytorch-24db27c8f9ec
        self.dim = dim
        self.d_k = dim // n_heads
        self.n_heads = n_heads

        self.q_linear = nn.Linear(dim, dim, bias = False)
        self.k_linear = nn.Linear(dim, dim, bias = False)
        self.v_linear = nn.Linear(dim, dim, bias = False)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(dim, dim)


    def forward(
            self,
            q: torch.FloatTensor,
            k: torch.FloatTensor,
            v: torch.FloatTensor,
            key_padding_mask: torch.BoolTensor = None,
            attn_mask: torch.BoolTensor = None,
        ) -> torch.FloatTensor:
        """Computes the scaled multi-head attention form the input queries,
        keys and values.

        Project those queries, keys and values before feeding them
        to the `attention` function.

        The masks are boolean masks. Tokens are prevented to attends to
        positions where the mask is `True`.

        Args
        ----
            q: Batch of queries.
                Shape of [batch_size, seq_len_1, dim_model].
            k: Batch of keys.
                Shape of [batch_size, seq_len_2, dim_model].
            v: Batch of values.
                Shape of [batch_size, seq_len_2, dim_model].
            key_padding_mask: Prevent attending to padding tokens.
                Shape of [batch_size, seq_len_2].
            attn_mask: Prevent attending to subsequent tokens.
                Shape of [seq_len_1, seq_len_2].

        Output
        ------
            y: Computed multihead attention.
                Shape of [batch_size, seq_len_1, dim_model].
        """
        # TODO
        # Implement multi-head attention forward pass:
        # 1. Get batch_size = q.size(0)
        # 2. Handle None masks by creating all-ones tensors:
        #    if key_padding_mask is None: key_padding_mask = torch.ones((batch_size, k.size(1))).to(device)
        #    if attn_mask is None: attn_mask = torch.ones((q.size(1), k.size(1))).to(device)
        # 3. Combine masks: mask = attn_mask.logical_and(key_padding_mask).unsqueeze(1)
        # 4. Project and reshape Q, K, V using einops rearrange:
        #    q = rearrange(self.q_linear(q), 'b s (hnum k) -> b hnum s k', hnum=self.n_heads)
        #    k = rearrange(...)
        #    v = rearrange(...)
        # 5. Compute attention: y, attn = attention(q, k, v, mask, self.dropout)
        # 6. Merge heads: y = rearrange(y, 'b hnum s v -> b s (hnum v)')
        # 7. Apply output projection: y = self.out(y)
        # 8. Return y
        # See: Lynn-Evans, Samuel. (2018). How to code The Transformer in Pytorch. https://towardsdatascience.com/how-to-code-the-transformer-in-pytorch-24db27c8f9ec
        # Einops. (unspecified). Writing a better code with pytorch and einops. https://einops.rocks/pytorch-examples.html

        batch_size = q.size(0)

        if key_padding_mask is None:
            key_padding_mask = torch.ones((batch_size, 1, k.size(1))).to(device)  # [B, 1, S]
        if attn_mask is None:
            attn_mask = torch.ones((1, q.size(1), k.size(1))).to(device)          # [1, S, S]

        # [1, S, S] & [B, 1, S] -> [B, S, S] -> unsqueeze(1) -> [B, 1, S, S]
        mask = attn_mask.logical_and(key_padding_mask).unsqueeze(1)
        q = rearrange(self.q_linear(q), 'b s (h d) -> b h s d', h=self.n_heads)
        k = rearrange(self.k_linear(k), 'b s (h d) -> b h s d', h=self.n_heads)
        v = rearrange(self.v_linear(v), 'b s (h d) -> b h s d', h=self.n_heads)
        y, attn = attention(q, k, v, mask, self.dropout)
        y = rearrange(y, 'b h s d -> b s (h d)')
        y = self.out(y)
        return y



### Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    """
    This PE module comes from:
    Pytorch. (2021). LANGUAGE MODELING WITH NN.TRANSFORMER AND TORCHTEXT. https://pytorch.org/tutorials/beginner/transformer_tutorial.html
    """
    def __init__(self, d_model: int, dropout: float, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        position = torch.arange(max_len).unsqueeze(1).to(device)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)).to(device)
        pe = torch.zeros(max_len, 1, d_model).to(device)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = rearrange(x, "b s e -> s b e")
        """
        Args:
            x: Tensor, shape [seq_len, batch_size, embedding_dim]
        """
        x = x + self.pe[:x.size(0)]
        x = rearrange(x, "s b e -> b s e")
        return self.dropout(x)

### Encoder and decoder layers

**TranformerEncoder**

Apply self-attention layers onto the source tokens.
It only needs the source key padding mask.


**TranformerDecoder**

Apply masked self-attention layers to the target tokens and cross-attention
layers between the source and the target tokens.
It needs the source and target key padding masks, and the target attention mask.

In [ ]:
class FeedForward(nn.Module):
    """
    Simple feedforward module used in the transformer model
    Inspired from: https://towardsdatascience.com/how-to-code-the-transformer-in-pytorch-24db27c8f9ec
    """
    def __init__(
        self,
        d_model: int,
        d_ff: int,
        dropout: float
      ):
      super().__init__()

      self.linear_1 = nn.Linear(d_model, d_ff)
      self.linear_2 = nn.Linear(d_ff, d_model)
      self.dropout_1 = nn.Dropout(dropout)
      self.dropout_2 = nn.Dropout(dropout)

    def forward(self, x):
      x = self.linear_1(x)
      x = F.relu(x)
      x = self.dropout_1(x)
      x = self.linear_2(x)
      x = self.dropout_2(x)
      return x


class TransformerDecoderLayer(nn.Module):
    """Single decoder layer.

    Parameters
    ----------
        d_model: The dimension of decoders inputs/outputs.
        dim_feedforward: Hidden dimension of the feedforward networks.
        nheads: Number of heads for each multi-head attention.
        dropout: Dropout rate.
    """

    def __init__(
            self,
            d_model: int,
            d_ff: int,
            nhead: int,
            dropout: float
        ):
        super().__init__()

        # TODO
        # Initialize decoder layer components:
        # 1. Masked self-attention: self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        #    (target attends to previous target tokens only)
        # 2. Cross-attention: self.enc_dec_attn = MultiheadAttention(d_model, nhead, dropout)
        #    (target queries attend to encoder output keys/values)
        # 3. Feedforward network: self.ff = FeedForward(d_model, d_ff, dropout)
        # 4. Dropout layers: self.dropout_1 = nn.Dropout(dropout), self.dropout_2 = nn.Dropout(dropout)
        # 5. Layer normalization: self.norm_1, self.norm_2, self.norm_3 = nn.LayerNorm(d_model)

        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.enc_dec_attn = MultiheadAttention(d_model, nhead, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.dropout_1 = nn.Dropout(dropout)
        self.dropout_2 = nn.Dropout(dropout)
        self.norm_1 = nn.LayerNorm(d_model)
        self.norm_2 = nn.LayerNorm(d_model)
        self.norm_3 = nn.LayerNorm(d_model)

    def forward(
            self,
            tgt: torch.FloatTensor,
            memory: torch.FloatTensor,
            tgt_mask_attn: torch.BoolTensor,
            memory_key_padding_mask: torch.BoolTensor,
            tgt_key_padding_mask: torch.BoolTensor,
        ) -> torch.FloatTensor:
        """Decode the next target tokens based on the previous tokens.

        Args
        ----
            src: Batch of source sentences.
                Shape of [batch_size, src_seq_len, dim_model].
            tgt: Batch of target sentences.
                Shape of [batch_size, tgt_seq_len, dim_model].
            tgt_mask_attn: Mask to prevent attention to subsequent tokens.
                Shape of [tgt_seq_len, tgt_seq_len].
            memory_key_padding_mask: Mask to prevent attention to padding in src sequence.
                Shape of [batch_size, src_seq_len].
            tgt_key_padding_mask: Mask to prevent attention to padding in tgt sequence.
                Shape of [batch_size, tgt_seq_len].

        Output
        ------
            y:  Batch of sequence of embeddings representing the predicted target tokens
                Shape of [batch_size, tgt_seq_len, dim_model].
        """
        x = tgt
        x = self.norm_1(x + self.dropout_1(self.self_attn(x, x, x, attn_mask=tgt_mask_attn, key_padding_mask=tgt_key_padding_mask)))
        x = self.norm_2(x + self.dropout_2(self.enc_dec_attn.forward(x, memory, memory, key_padding_mask=memory_key_padding_mask)))
        x = self.norm_3(x + self.ff(x))

        return x


class TransformerDecoder(nn.Module):
    """Implementation of the transformer decoder stack.

    Parameters
    ----------
        d_model: The dimension of decoders inputs/outputs.
        dim_feedforward: Hidden dimension of the feedforward networks.
        num_decoder_layers: Number of stacked decoders.
        nheads: Number of heads for each multi-head attention.
        dropout: Dropout rate.
    """

    def __init__(
            self,
            d_model: int,
            d_ff: int,
            num_decoder_layer:int ,
            nhead: int,
            dropout: float
        ):
        super().__init__()

        decoder_layer = TransformerDecoderLayer(d_model, d_ff, nhead, dropout)
        self.decoder_layers = clones(decoder_layer, num_decoder_layer)

    def forward(
            self,
            tgt: torch.FloatTensor,
            memory: torch.FloatTensor,
            tgt_mask_attn: torch.BoolTensor,
            memory_key_padding_mask: torch.BoolTensor,
            tgt_key_padding_mask: torch.BoolTensor,
        ) -> torch.FloatTensor:
        """Decodes the source sequence by sequentially passing.
        the encoded source sequence and the target sequence through the decoder stack.

        Args
        ----
            src: Batch of encoded source sentences.
                Shape of [batch_size, src_seq_len, dim_model].
            tgt: Batch of taget sentences.
                Shape of [batch_size, tgt_seq_len, dim_model].
            tgt_mask_attn: Mask to prevent attention to subsequent tokens.
                Shape of [tgt_seq_len, tgt_seq_len].
            memory_key_padding_mask: Mask to prevent attention to padding in src sequence.
                Shape of [batch_size, src_seq_len].
            tgt_key_padding_mask: Mask to prevent attention to padding in tgt sequence.
                Shape of [batch_size, tgt_seq_len].

        Output
        ------
            y:  Batch of sequence of embeddings representing the predicted target tokens
                Shape of [batch_size, tgt_seq_len, dim_model].
        """
        out = tgt
        for layer in self.decoder_layers:
          out = layer(out, memory, tgt_mask_attn=tgt_mask_attn, tgt_key_padding_mask=tgt_key_padding_mask, memory_key_padding_mask=memory_key_padding_mask)
        return out


class TransformerEncoderLayer(nn.Module):
    """Single encoder layer.

    Parameters
    ----------
        d_model: The dimension of input tokens.
        dim_feedforward: Hidden dimension of the feedforward networks.
        nheads: Number of heads for each multi-head attention.
        dropout: Dropout rate.
    """

    def __init__(
            self,
            d_model: int,
            d_ff: int,
            nhead: int,
            dropout: float,
        ):
        super().__init__()

        # TODO
        # Initialize encoder layer components:
        # 1. Self-attention: self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        # 2. Position-wise feedforward: self.ff = FeedForward(d_model, d_ff, dropout)
        # 3. Dropout layer: self.dropout = nn.Dropout(dropout)
        # 4. Layer normalization: self.norm_1 = nn.LayerNorm(d_model), self.norm_2 = nn.LayerNorm(d_model)
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)

        self.dropout = nn.Dropout(dropout)
        self.norm_1 = nn.LayerNorm(d_model)
        self.norm_2 = nn.LayerNorm(d_model)

    def forward(
        self,
        src: torch.FloatTensor,
        key_padding_mask: torch.BoolTensor
        ) -> torch.FloatTensor:
        """Encodes the input. Does not attend to masked inputs.

        Args
        ----
            src: Batch of embedded source tokens.
                Shape of [batch_size, src_seq_len, dim_model].
            key_padding_mask: Mask preventing attention to padding tokens.
                Shape of [batch_size, src_seq_len].

        Output
        ------
            y: Batch of encoded source tokens.
                Shape of [batch_size, src_seq_len, dim_model].
        """
        x = src
        x = self.norm_1(x + self.dropout(self.self_attn.forward(x, x, x, key_padding_mask=key_padding_mask)))
        x = self.norm_2(x + self.ff(x))
        return x


class TransformerEncoder(nn.Module):
    """Implementation of the transformer encoder stack.

    Parameters
    ----------
        d_model: The dimension of encoders inputs.
        dim_feedforward: Hidden dimension of the feedforward networks.
        num_encoder_layers: Number of stacked encoders.
        nheads: Number of heads for each multi-head attention.
        dropout: Dropout rate.
    """

    def __init__(
            self,
            d_model: int,
            dim_feedforward: int,
            num_encoder_layers: int,
            nheads: int,
            dropout: float
        ):
        super().__init__()

        encoder_layer = TransformerEncoderLayer(d_model, dim_feedforward, nheads, dropout)
        self.encoder_layers = clones(encoder_layer, num_encoder_layers)


    def forward(
            self,
            src: torch.FloatTensor,
            key_padding_mask: torch.BoolTensor
        ) -> torch.FloatTensor:
        """Encodes the source sequence by sequentially passing.
        the source sequence through the encoder stack.

        Args
        ----
            src: Batch of embedded source sentences.
                Shape of [batch_size, src_seq_len, dim_model].
            key_padding_mask: Mask preventing attention to padding tokens.
                Shape of [batch_size, src_seq_len].

        Output
        ------
            y: Batch of encoded source sequence.
                Shape of [batch_size, src_seq_len, dim_model].
        """
        out = src
        for layer in self.encoder_layers:
          out = layer(out, key_padding_mask=key_padding_mask)
        return out

### Main layers
This section gather the `Transformer` and the `TranslationTransformer` modules.

**Transformer**


The classical transformer architecture.
It takes the source and target tokens embeddings and
do the forward pass through the encoder and decoder.

**Translation Transformer**

Compute the source and target tokens embeddings, and apply a final head to produce next token logits.
The output must not be the softmax but just the logits, because we use the `nn.CrossEntropyLoss`.

It also creates the *src_key_padding_mask*, the *tgt_key_padding_mask* and the *tgt_mask_attn*.

In [ ]:
import math
class Transformer(nn.Module):
    """Implementation of a Transformer based on the paper: https://arxiv.org/pdf/1706.03762.pdf.

    Parameters
    ----------
        d_model: The dimension of encoders/decoders inputs/ouputs.
        nhead: Number of heads for each multi-head attention.
        num_encoder_layers: Number of stacked encoders.
        num_decoder_layers: Number of stacked encoders.
        dim_feedforward: Hidden dimension of the feedforward networks.
        dropout: Dropout rate.
    """

    def __init__(
            self,
            d_model: int,
            nhead: int,
            num_encoder_layers: int,
            num_decoder_layers: int,
            dim_feedforward: int,
            dropout: float,
        ):
        super().__init__()

        self.d_model = d_model
        self.encoder = TransformerEncoder(d_model, dim_feedforward, num_encoder_layers, nhead, dropout)
        self.decoder = TransformerDecoder(d_model, dim_feedforward, num_decoder_layers, nhead, dropout)

    def forward(
            self,
            src: torch.FloatTensor,
            tgt: torch.FloatTensor,
            tgt_mask_attn: torch.BoolTensor,
            src_key_padding_mask: torch.BoolTensor,
            tgt_key_padding_mask: torch.BoolTensor
        ) -> torch.FloatTensor:
        """Compute next token embeddings.

        Args
        ----
            src: Batch of source sequences.
                Shape of [batch_size, src_seq_len, dim_model].
            tgt: Batch of target sequences.
                Shape of [batch_size, tgt_seq_len, dim_model].
            tgt_mask_attn: Mask to prevent attention to subsequent tokens.
                Shape of [tgt_seq_len, tgt_seq_len].
            src_key_padding_mask: Mask to prevent attention to padding in src sequence.
                Shape of [batch_size, src_seq_len].
            tgt_key_padding_mask: Mask to prevent attention to padding in tgt sequence.
                Shape of [batch_size, tgt_seq_len].

        Output
        ------
            y: Next token embeddings, given the previous target tokens and the source tokens.
                Shape of [batch_size, tgt_seq_len, dim_model].
        """
        out = self.encoder.forward(src, src_key_padding_mask)
        out = self.decoder.forward(tgt, out, tgt_mask_attn, src_key_padding_mask, tgt_key_padding_mask)
        return out


class TranslationTransformer(nn.Module):
    """Basic Transformer encoder and decoder for a translation task.
    Manage the masks creation, and the token embeddings.
    Position embeddings can be learnt with a standard `nn.Embedding` layer.

    Parameters
    ----------
        n_tokens_src: Number of tokens in the source vocabulary.
        n_tokens_tgt: Number of tokens in the target vocabulary.
        n_heads: Number of heads for each multi-head attention.
        dim_embedding: Dimension size of the word embeddings (for both language).
        dim_hidden: Dimension size of the feedforward layers
            (for both the encoder and the decoder).
        n_layers: Number of layers in the encoder and decoder.
        dropout: Dropout rate.
        src_pad_idx: Source padding index value.
        tgt_pad_idx: Target padding index value.
    """
    def __init__(
            self,
            n_tokens_src: int,
            n_tokens_tgt: int,
            n_heads: int,
            dim_embedding: int,
            dim_hidden: int,
            n_layers: int,
            dropout: float,
            src_pad_idx: int,
            tgt_pad_idx: int,
        ):
        super().__init__()

        self.dim_embedding = dim_embedding
        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx

        self.pe_src = PositionalEncoding(dim_embedding, dropout)
        self.pe_tgt = PositionalEncoding(dim_embedding, dropout)

        self.embedding_src = nn.Embedding(n_tokens_src, dim_embedding)
        self.embedding_tgt = nn.Embedding(n_tokens_tgt, dim_embedding)

        self.my_transformer = Transformer(dim_embedding, n_heads, n_layers, n_layers, dim_hidden, dropout)
        self.out_layer = nn.Linear(dim_embedding, n_tokens_tgt)


    def forward(
            self,
            source: torch.LongTensor,
            target: torch.LongTensor
        ) -> torch.FloatTensor:
        """Predict the target tokens based on the source tokens.

        Args
        ----
            source: Batch of source sentences.
                Shape of [batch_size, seq_len_src].
            target: Batch of target sentences.
                Shape of [batch_size, seq_len_tgt].

        Output
        ------
            y: Batch of predictions of the next token distributions in the target sentences.
                Shape of [batch_size, seq_len_tgt, n_tokens_tgt].
        """
        # TODO
        # Implement full translation forward pass:
        # 1. Build causal mask for target: tgt_mask = self.build_mask(target.size(1))
        #    This creates upper triangular mask preventing attention to future positions
        # 2. Build padding masks:
        #    src_key_padding_mask = self.build_pad_mask(source, self.src_pad_idx)
        #    tgt_key_padding_mask = self.build_pad_mask(target, self.tgt_pad_idx)
        # 3. Embed and scale source tokens:
        #    source = self.embedding_src(source) * Scaling by sqrt(d_model) as per original Transformer paper
        # 4. Add positional encoding to source: source = self.pe_src(source)
        # 5. Embed and scale target: target = self.embedding_tgt(target) * math.sqrt(self.dim_embedding)
        # 6. Add positional encoding to target: target = self.pe_tgt(target)
        # 7. Pass through transformer:
        #    trans_out = self.my_transformer(src=source, tgt=target, tgt_mask_attn=tgt_mask,
        #                                     src_key_padding_mask=src_key_padding_mask,
        #                                     tgt_key_padding_mask=tgt_key_padding_mask)
        # 8. Project to vocabulary: y = self.out_layer(trans_out)
        # 9. Return y with shape [batch_size, seq_len_tgt, n_tokens_tgt]

        tgt_mask = self.build_mask(target.size(1))
        src_key_padding_mask = self.build_pad_mask(source, self.src_pad_idx)
        tgt_key_padding_mask = self.build_pad_mask(target, self.tgt_pad_idx)
        source = self.embedding_src(source) * math.sqrt(self.dim_embedding)
        source = self.pe_src(source)
        target = self.embedding_tgt(target) * math.sqrt(self.dim_embedding)
        target = self.pe_tgt(target)
        trans_out = self.my_transformer(src=source, tgt=target, tgt_mask_attn=tgt_mask,
                                        src_key_padding_mask=src_key_padding_mask,
                                        tgt_key_padding_mask=tgt_key_padding_mask)
        y = self.out_layer(trans_out)
        return y



    def build_mask(self, size):
        """
        From:
        MELCHOR, Daniel. (2021). A detailed guide to PyTorch’s nn.Transformer() module. https://towardsdatascience.com/a-detailed-guide-to-pytorchs-nn-transformer-module-c80afbc9ffb1
        """
        attn_shape = (1, size, size)
        mask = np.triu(np.ones(attn_shape), k=1).astype('uint8')
        return (torch.from_numpy(mask) == 0).to(device)

    def build_pad_mask(self, tensor, pad_idx):
        mask = (tensor != pad_idx).unsqueeze(-2)
        return mask


# Greedy search

Here you have to implement a geedy search to generate a target translation from a trained model and an input source string.
The next token will simply be the most probable one.

In [ ]:
def indices_terminated(
        target: torch.FloatTensor,
        eos_token: int
    ) -> tuple:
    """Split the target sentences between the terminated and the non-terminated
    sentence. Return the indices of those two groups.

    Args
    ----
        target: The sentences.
            Shape of [batch_size, n_tokens].
        eos_token: Value of the End-of-Sentence token.

    Output
    ------
        terminated: Indices of the terminated sentences (who's got the eos_token).
            Shape of [n_terminated, ].
        non-terminated: Indices of the unfinished sentences.
            Shape of [batch_size-n_terminated, ].
    """
    terminated = [i for i, t in enumerate(target) if eos_token in t]
    non_terminated = [i for i, t in enumerate(target) if eos_token not in t]
    return torch.LongTensor(terminated), torch.LongTensor(non_terminated)

In [ ]:
def greedy_search(
        model: nn.Module,
        source: str,
        src_vocab: Vocab,
        tgt_vocab: Vocab,
        src_tokenizer,
        device: str,
        max_sentence_length: int,
    ) -> str:
    """Do a greedy search to produce probable translations.

    Args
    ----
        model: The translation model. Assumes it produces logits score (before softmax).
        source: The sentence to translate.
        src_vocab: The source vocabulary.
        tgt_vocab: The target vocabulary.
        device: Device to which we make the inference.
        max_sentence_length: Maximum number of tokens for the translated sentence.

    Output
    ------
        sentence: The translated source sentence.
    """
    src_tokens = ['<bos>'] + src_tokenizer(source) + ['<eos>']
    src_tokens = src_vocab(src_tokens)

    tgt_tokens = ['<bos>']
    tgt_tokens = tgt_vocab(tgt_tokens)

    # To tensor and add unitary batch dimension
    src_tokens = torch.LongTensor(src_tokens).to(device)
    tgt_tokens = torch.LongTensor(tgt_tokens).unsqueeze(dim=0).to(device)
    target_probs = torch.FloatTensor([1]).to(device)
    model.to(device)
    EOS_IDX = tgt_vocab['<eos>']

    with torch.no_grad():
      while tgt_tokens.shape[1] < max_sentence_length:

            batch_size, n_tokens = tgt_tokens.shape

            src = einops.repeat(src_tokens, 't -> b t', b=tgt_tokens.shape[0])
            predicted = model.forward(src, tgt_tokens)
            predicted = torch.softmax(predicted, dim=-1)

            probs, predicted = predicted[:, -1].topk(k=1, dim=-1)

            # Separe between terminated sentences and the others
             # Separe between terminated sentences and the others
            idx_terminated, idx_not_terminated = indices_terminated(tgt_tokens, EOS_IDX)
            idx_terminated, idx_not_terminated = idx_terminated.to(device), idx_not_terminated.to(device)

            tgt_terminated = torch.index_select(tgt_tokens, dim=0, index=idx_terminated)
            tgt_probs_terminated = torch.index_select(target_probs, dim=0, index=idx_terminated)

            filter_t = lambda t: torch.index_select(t, dim=0, index=idx_not_terminated)
            tgt_others = filter_t(tgt_tokens)
            if tgt_others.shape[0] == 0:
              break
            tgt_probs_others = filter_t(target_probs)
            predicted = filter_t(predicted)
            probs = filter_t(probs)

            # Add padding to terminated target
            padd = torch.zeros((len(tgt_terminated), 1), dtype=torch.long, device=device)
            tgt_terminated = torch.cat(
                (tgt_terminated, padd),
                dim=1
            )

            # Update each target sentence probabilities

            tgt_probs_others *= probs.flatten()
            tgt_probs_terminated *= 0.999  # Penalize short sequences overtime

            # Group up the terminated and the others
            target_probs = torch.cat(
                (tgt_probs_others, tgt_probs_terminated),
                dim=0
            )

            n_tokens = tgt_others.shape[1]

            tgt_others = torch.cat((tgt_others, predicted), dim=-1)
            tgt_others = tgt_others.view(batch_size, n_tokens+1)


            tgt_tokens = torch.cat(
                (tgt_others, tgt_terminated),
                dim=0
            )


            target_probs, indices = target_probs.topk(k=1, dim=0)
            tgt_tokens = torch.index_select(tgt_tokens, dim=0, index=indices)


    sentences = []
    for tgt_sentence in tgt_tokens:
        tgt_sentence = list(tgt_sentence)[1:]  # Remove <bos> token
        tgt_sentence = list(takewhile(lambda t: t != EOS_IDX, tgt_sentence))
        tgt_sentence = ' '.join(tgt_vocab.lookup_tokens(tgt_sentence))
        sentences.append(tgt_sentence)

    sentences = [beautify(s) for s in sentences]

    # Join the sentences with their likelihood
    sentences = [(s, p.item()) for s, p in zip(sentences, target_probs)]
    # Sort the sentences by their likelihood
    sentences = [(s, p) for s, p in sorted(sentences, key=lambda k: k[1], reverse=True)]

    return sentences



# Beam search
Beam search is a smarter way of producing a sequence of tokens from
an autoregressive model than just using a greedy search.

The greedy search always choose the most probable token as the unique
and only next target token, and repeat this processus until the *\<eos\>* token is predicted.

Instead, the beam search selects the k-most probable tokens at each step.
From those k tokens, the current sequence is duplicated k times and the k tokens are appended to the k sequences to produce new k sequences.

*You don't have to understand this code, but understanding this code once the TP is over could improve your torch tensors skills.*

---

**More explanations**

Since it is done at each step, the number of sequences grows exponentially (k sequences after the first step, k² sequences after the second...).
In order to keep the number of sequences low, we remove sequences except the top-s most likely sequences.
To do that, we keep track of the likelihood of each sequence.

Formally, we define $s = [s_1, ..., s_{N_s}]$ as the source sequence made of $N_s$ tokens.
We also define $t^i = [t_1, ..., t_i]$ as the target sequence at the beginning of the step $i$.

The output of the model parameterized by $\theta$ is:

$$
T_{i+1} = p(t_{i+1} | s, t^i ; \theta )
$$

Where $T_{i+1}$ is the distribution of the next token $t_{i+1}$.

Then, we define the likelihood of a target sentence $t = [t_1, ..., t_{N_t}]$ as:

$$
L(t) = \prod_{i=1}^{N_t - 1} p(t_{i+1} | s, t_{i}; \theta )
$$

Pseudocode of the beam search:
```
source: [N_s source tokens]  # Shape of [total_source_tokens]
target: [1, <bos> token]  # Shape of [n_sentences, current_target_tokens]
target_prob: [1]  # Shape of [n_sentences]
# We use `n_sentences` as the batch_size dimension

while current_target_tokens <= max_target_length:
    source = repeat(source, n_sentences)  # Shape of [n_sentences, total_source_tokens]
    predicted = model(source, target)[:, -1]  # Predict the next token distributions of all the n_sentences
    tokens_idx, tokens_prob = topk(predicted, k)

    # Append the `n_sentences * k` tokens to the `n_sentences` sentences
    target = repeat(target, k)  # Shape of [n_sentences * k, current_target_tokens]
    target = append_tokens(target, tokens_idx)  # Shape of [n_sentences * k, current_target_tokens + 1]

    # Update the sentences probabilities
    target_prob = repeat(target_prob, k)  # Shape of [n_sentences * k]
    target_prob *= tokens_prob

    if n_sentences * k >= max_sentences:
        target, target_prob = topk_prob(target, target_prob, k=max_sentences)
    else:
        n_sentences *= k

    current_target_tokens += 1
```

In [ ]:
def beautify(sentence: str) -> str:
    """Removes useless spaces.
    """
    punc = {'.', ',', ';'}
    for p in punc:
        sentence = sentence.replace(f' {p}', p)

    links = {'-', "'"}
    for l in links:
        sentence = sentence.replace(f'{l} ', l)
        sentence = sentence.replace(f' {l}', l)

    return sentence

In [ ]:
def append_beams(
        target: torch.FloatTensor,
        beams: torch.FloatTensor
    ) -> torch.FloatTensor:
    """Add the beam tokens to the current sentences.
    Duplicate the sentences so one token is added per beam per batch.

    Args
    ----
        target: Batch of unfinished sentences.
            Shape of [batch_size, n_tokens].
        beams: Batch of beams for each sentences.
            Shape of [batch_size, n_beams].

    Output
    ------
        target: Batch of sentences with one beam per sentence.
            Shape of [batch_size * n_beams, n_tokens+1].
    """
    batch_size, n_beams = beams.shape
    n_tokens = target.shape[1]

    target = einops.repeat(target, 'b t -> b c t', c=n_beams)  # [batch_size, n_beams, n_tokens]
    beams = beams.unsqueeze(dim=2)  # [batch_size, n_beams, 1]

    target = torch.cat((target, beams), dim=2)  # [batch_size, n_beams, n_tokens+1]
    target = target.view(batch_size*n_beams, n_tokens+1)  # [batch_size * n_beams, n_tokens+1]
    return target


def beam_search(
        model: nn.Module,
        source: str,
        src_vocab: Vocab,
        tgt_vocab: Vocab,
        src_tokenizer,
        device: str,
        beam_width: int,
        max_target: int,
        max_sentence_length: int,
    ) -> list:
    """Do a beam search to produce probable translations.

    Args
    ----
        model: The translation model. Assumes it produces linear score (before softmax).
        source: The sentence to translate.
        src_vocab: The source vocabulary.
        tgt_vocab: The target vocabulary.
        device: Device to which we make the inference.
        beam_width: Number of top-k tokens we keep at each stage.
        max_target: Maximum number of target sentences we keep at the end of each stage.
        max_sentence_length: Maximum number of tokens for the translated sentence.

    Output
    ------
        sentences: List of sentences orderer by their likelihood.
    """
    src_tokens = ['<bos>'] + src_tokenizer(source) + ['<eos>']
    src_tokens = src_vocab(src_tokens)

    tgt_tokens = ['<bos>']
    tgt_tokens = tgt_vocab(tgt_tokens)

    # To tensor and add unitary batch dimension
    src_tokens = torch.LongTensor(src_tokens).to(device)
    tgt_tokens = torch.LongTensor(tgt_tokens).unsqueeze(dim=0).to(device)
    target_probs = torch.FloatTensor([1]).to(device)
    model.to(device)

    EOS_IDX = tgt_vocab['<eos>']
    with torch.no_grad():
        while tgt_tokens.shape[1] < max_sentence_length:
            batch_size, n_tokens = tgt_tokens.shape

            # Get next beams
            src = einops.repeat(src_tokens, 't -> b t', b=tgt_tokens.shape[0])
            predicted = model.forward(src, tgt_tokens)
            predicted = torch.softmax(predicted, dim=-1)
            probs, predicted = predicted[:, -1].topk(k=beam_width, dim=-1)

            # Separe between terminated sentences and the others
            idx_terminated, idx_not_terminated = indices_terminated(tgt_tokens, EOS_IDX)
            idx_terminated, idx_not_terminated = idx_terminated.to(device), idx_not_terminated.to(device)

            tgt_terminated = torch.index_select(tgt_tokens, dim=0, index=idx_terminated)
            tgt_probs_terminated = torch.index_select(target_probs, dim=0, index=idx_terminated)

            filter_t = lambda t: torch.index_select(t, dim=0, index=idx_not_terminated)
            tgt_others = filter_t(tgt_tokens)
            tgt_probs_others = filter_t(target_probs)
            predicted = filter_t(predicted)
            probs = filter_t(probs)

            # Add the top tokens to the previous target sentences
            tgt_others = append_beams(tgt_others, predicted)

            # Add padding to terminated target
            padd = torch.zeros((len(tgt_terminated), 1), dtype=torch.long, device=device)
            tgt_terminated = torch.cat(
                (tgt_terminated, padd),
                dim=1
            )

            # Update each target sentence probabilities
            tgt_probs_others = torch.repeat_interleave(tgt_probs_others, beam_width)
            tgt_probs_others *= probs.flatten()
            tgt_probs_terminated *= 0.999  # Penalize short sequences overtime

            # Group up the terminated and the others
            target_probs = torch.cat(
                (tgt_probs_others, tgt_probs_terminated),
                dim=0
            )


            tgt_tokens = torch.cat(
                (tgt_others, tgt_terminated),
                dim=0
            )

            # Keep only the top `max_target` target sentences
            if target_probs.shape[0] <= max_target:
                continue

            target_probs, indices = target_probs.topk(k=max_target, dim=0)
            tgt_tokens = torch.index_select(tgt_tokens, dim=0, index=indices)

    sentences = []
    for tgt_sentence in tgt_tokens:
        tgt_sentence = tgt_sentence[1:].tolist()  # Remove <bos> token and convert to Python list
        tgt_sentence = list(takewhile(lambda t: t != EOS_IDX, tgt_sentence))
        tgt_sentence = ' '.join(tgt_vocab.lookup_tokens(tgt_sentence))
        sentences.append(tgt_sentence)

    sentences = [beautify(s) for s in sentences]

    # Join the sentences with their likelihood
    sentences = [(s, p.item()) for s, p in zip(sentences, target_probs)]
    # Sort the sentences by their likelihood
    sentences = [(s, p) for s, p in sorted(sentences, key=lambda k: k[1], reverse=True)]

    return sentences

# Training loop
This is a basic training loop code. It takes a big configuration dictionnary to avoid never ending arguments in the functions.
We use [Weights and Biases](https://wandb.ai/) to log the trainings.
It logs every training informations and model performances in the cloud.
You have to create an account to use it. Every accounts are free for individuals or research teams.

In [ ]:
def print_logs(dataset_type: str, logs: dict):
    """Print the logs.

    Args
    ----
        dataset_type: Either "Train", "Eval", "Test" type.
        logs: Containing the metric's name and value.
    """
    desc = [
        f'{name}: {value:.2f}'
        for name, value in logs.items()
    ]
    desc = '\t'.join(desc)
    desc = f'{dataset_type} -\t' + desc
    desc = desc.expandtabs(5)
    print(desc)


def topk_accuracy(
        real_tokens: torch.FloatTensor,
        probs_tokens: torch.FloatTensor,
        k: int,
        tgt_pad_idx: int,
    ) -> torch.FloatTensor:
    """Compute the top-k accuracy.
    We ignore the PAD tokens.

    Args
    ----
        real_tokens: Real tokens of the target sentence.
            Shape of [batch_size * n_tokens].
        probs_tokens: Tokens probability predicted by the model.
            Shape of [batch_size * n_tokens, n_target_vocabulary].
        k: Top-k accuracy threshold.
        src_pad_idx: Source padding index value.

    Output
    ------
        acc: Scalar top-k accuracy value.
    """
    total = (real_tokens != tgt_pad_idx).sum()

    _, pred_tokens = probs_tokens.topk(k=k, dim=-1)  # [batch_size * n_tokens, k]
    real_tokens = einops.repeat(real_tokens, 'b -> b k', k=k)  # [batch_size * n_tokens, k]

    good = (pred_tokens == real_tokens) & (real_tokens != tgt_pad_idx)
    acc = good.sum() / total
    return acc


def loss_batch(
        model: nn.Module,
        source: torch.LongTensor,
        target: torch.LongTensor,
        config: dict,
    )-> dict:
    """Compute the metrics associated with this batch.
    The metrics are:
        - loss
        - top-1 accuracy
        - top-5 accuracy
        - top-10 accuracy

    Args
    ----
        model: The model to train.
        source: Batch of source tokens.
            Shape of [batch_size, n_src_tokens].
        target: Batch of target tokens.
            Shape of [batch_size, n_tgt_tokens].
        config: Additional parameters.

    Output
    ------
        metrics: Dictionnary containing evaluated metrics on this batch.
    """
    device = config['device']
    loss_fn = config['loss'].to(device)
    metrics = dict()

    source, target = source.to(device), target.to(device)
    target_in, target_out = target[:, :-1], target[:, 1:]

    # Loss
    pred = model(source, target_in)  # [batch_size, n_tgt_tokens-1, n_vocab]
    pred = pred.view(-1, pred.shape[2])  # [batch_size * (n_tgt_tokens - 1), n_vocab]
    target_out = target_out.flatten()  # [batch_size * (n_tgt_tokens - 1),]
    metrics['loss'] = loss_fn(pred, target_out)

    # Accuracy - we ignore the padding predictions
    for k in [1, 5, 10]:
        metrics[f'top-{k}'] = topk_accuracy(target_out, pred, k, config['tgt_pad_idx'])

    return metrics


def eval_model(model: nn.Module, dataloader: DataLoader, config: dict) -> dict:
    """Evaluate the model on the given dataloader.
    """
    device = config['device']
    logs = defaultdict(list)

    model.to(device)
    model.eval()

    with torch.no_grad():
        for source, target in dataloader:
            metrics = loss_batch(model, source, target, config)
            for name, value in metrics.items():
                logs[name].append(value.cpu().item())

    for name, values in logs.items():
        logs[name] = np.mean(values)
    return logs


def train_model(model: nn.Module, config: dict):
    """Train the model in a teacher forcing manner.
    """
    train_loader, val_loader = config['train_loader'], config['val_loader']
    train_dataset, val_dataset = train_loader.dataset.dataset, val_loader.dataset.dataset
    optimizer = config['optimizer']
    clip = config['clip']
    device = config['device']

    columns = ['epoch']
    for mode in ['train', 'validation']:
        columns += [
            f'{mode} - {colname}'
            for colname in ['source', 'target', 'predicted', 'likelihood']
        ]
    log_table = wandb.Table(columns=columns)


    print(f'Starting training for {config["epochs"]} epochs, using {device}.')
    for e in range(config['epochs']):
        print(f'\nEpoch {e+1}')

        model.to(device)
        model.train()
        logs = defaultdict(list)

        for batch_id, (source, target) in enumerate(train_loader):
            optimizer.zero_grad()

            metrics = loss_batch(model, source, target, config)
            loss = metrics['loss']

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()

            for name, value in metrics.items():
                logs[name].append(value.cpu().item())  # Don't forget the '.item' to free the cuda memory

            if batch_id % config['log_every'] == 0:
                for name, value in logs.items():
                    logs[name] = np.mean(value)

                train_logs = {
                    f'Train - {m}': v
                    for m, v in logs.items()
                }
                wandb.log(train_logs)
                logs = defaultdict(list)

        # Logs
        if len(logs) != 0:
            for name, value in logs.items():
                logs[name] = np.mean(value)
            train_logs = {
                f'Train - {m}': v
                for m, v in logs.items()
            }
        else:
            logs = {
                m.split(' - ')[1]: v
                for m, v in train_logs.items()
            }

        print_logs('Train', logs)

        logs = eval_model(model, val_loader, config)
        print_logs('Eval', logs)
        val_logs = {
            f'Validation - {m}': v
            for m, v in logs.items()
        }

        val_source, val_target = val_dataset[ torch.randint(len(val_dataset), (1,)) ]
        val_pred, val_prob = beam_search(
            model,
            val_source,
            config['src_vocab'],
            config['tgt_vocab'],
            config['src_tokenizer'],
            device,  # It can take a lot of VRAM
            beam_width=10,
            max_target=100,
            max_sentence_length=config['max_sequence_length'],
        )[0]
        print(val_source)
        print(val_pred)

        logs = {**train_logs, **val_logs}  # Merge dictionnaries
        wandb.log(logs)  # Upload to the WandB cloud

        # Table logs
        train_source, train_target = train_dataset[ torch.randint(len(train_dataset), (1,)) ]
        train_pred, train_prob = beam_search(
            model,
            train_source,
            config['src_vocab'],
            config['tgt_vocab'],
            config['src_tokenizer'],
            device,  # It can take a lot of VRAM
            beam_width=10,
            max_target=100,
            max_sentence_length=config['max_sequence_length'],
        )[0]

        data = [
            e + 1,
            train_source, train_target, train_pred, train_prob,
            val_source, val_target, val_pred, val_prob,
        ]
        log_table.add_data(*data)

    # Log the table at the end of the training
    wandb.log({'Model predictions': log_table})

# Training the models
We can now finally train the models.
Choose the right hyperparameters, play with them and try to find
ones that lead to good models and good training curves.
Try to reach a loss under 1.0.

So you know, it is possible to get descent results with approximately 20 epochs.
With CUDA enabled, one epoch, even on a big model with a big dataset, shouldn't last more than 10 minutes.
A normal epoch is between 1 to 5 minutes.

*This is considering Colab Pro, we should try using free Colab to get better estimations.*

---

To test your implementations, it is easier to try your models
in a CPU instance. Indeed, Colab reduces your GPU instances priority
with the time you recently past using GPU instances. It would be
sad to consume all your GPU time on implementation testing.
Moreover, you should try your models on small datasets and with a small number of parameters.
For exemple, you could set:
```
MAX_SEQ_LEN = 10
MIN_TOK_FREQ = 20
dim_embedding = 40
dim_hidden = 60
n_layers = 1
```

You usually don't want to log anything onto WandB when testing your implementation.
To deactivate WandB without having to change any line of code, you can type `!wandb offline` in a cell.

Once you have rightly implemented the models, you can train bigger models on bigger datasets.
When you do this, do not forget to change the runtime as GPU (and use `!wandb online`)!

In [ ]:
# Checking GPU and logging to wandb (if desired)

# !wandb login

!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
# Instanciate the datasets

MAX_SEQ_LEN = 60
MIN_TOK_FREQ = 2
train_dataset, val_dataset = build_datasets(
    MAX_SEQ_LEN,
    MIN_TOK_FREQ,
    en_tokenizer,
    fr_tokenizer,
    train,
    valid,
)


print(f'English vocabulary size: {len(train_dataset.en_vocab):,}')
print(f'French vocabulary size: {len(train_dataset.fr_vocab):,}')

print(f'\nTraining examples: {len(train_dataset):,}')
print(f'Validation examples: {len(val_dataset):,}')

English vocabulary size: 9,196
French vocabulary size: 13,389

Training examples: 108,233
Validation examples: 24,053


In [ ]:
# Print vocabulary information
print("\n" + "="*60)
print("VOCABULARY INFORMATION")
print("="*60)

# Print vocabulary sizes
print(f"\nEnglish vocabulary size: {len(train_dataset.en_vocab):,}")
print(f"French vocabulary size: {len(train_dataset.fr_vocab):,}")

# Print special tokens
print("\nSpecial tokens:")
for special in ['<unk>', '<pad>', '<bos>', '<eos>']:
    en_idx = train_dataset.en_vocab[special]
    fr_idx = train_dataset.fr_vocab[special]
    print(f"  {special}: EN={en_idx}, FR={fr_idx}")

# Print first 20 tokens from each vocabulary
print("\nFirst 20 English tokens:")
for i in range(min(20, len(train_dataset.en_vocab))):
    token = train_dataset.en_vocab.lookup_token(i)
    print(f"  {i}: {token}")

print("\nFirst 20 French tokens:")
for i in range(min(20, len(train_dataset.fr_vocab))):
    token = train_dataset.fr_vocab.lookup_token(i)
    print(f"  {i}: {token}")


VOCABULARY INFORMATION

English vocabulary size: 9,196
French vocabulary size: 13,389

Special tokens:
  <unk>: EN=0, FR=0
  <pad>: EN=1, FR=1
  <bos>: EN=2, FR=2
  <eos>: EN=3, FR=3

First 20 English tokens:
  0: <unk>
  1: <pad>
  2: <bos>
  3: <eos>
  4: What
  5: an
  6: opportunity
  7: !
  8: Our
  9: sales
  10: are
  11: decreasing
  12: .
  13: You
  14: should
  15: rest
  16: a
  17: little
  18: Someone
  19: stole

First 20 French tokens:
  0: <unk>
  1: <pad>
  2: <bos>
  3: <eos>
  4: Quelle
  5: occasion
  6: !
  7: Nos
  8: ventes
  9: .
  10: Tu
  11: devrais
  12: te
  13: reposer
  14: un
  15: peu
  16: Quelqu'
  17: a
  18: volé
  19: mon


In [ ]:
# Build the model, the dataloaders, optimizer and the loss function
# Log every hyperparameters and arguments into the config dictionnary
config = {
    # General parameters
    'epochs': 20,
    'batch_size': 128,
    'lr': 1e-3,
    'betas': (0.9, 0.99),
    'clip': 5,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',

    # Model parameters
    'n_tokens_src': len(train_dataset.en_vocab),
    'n_tokens_tgt': len(train_dataset.fr_vocab),
    'n_heads': 4,
    'dim_embedding': 196,
    'dim_hidden': 256,
    'n_layers': 3,
    'dropout': 0.1,
    #'model_type': 'GRU',
    #'model_type': 'Transformer',
    'model_type': 'RNN',


    # Others
    'max_sequence_length': MAX_SEQ_LEN,
    'min_token_freq': MIN_TOK_FREQ,
    'src_vocab': train_dataset.en_vocab,
    'tgt_vocab': train_dataset.fr_vocab,
    'src_tokenizer': en_tokenizer,
    'tgt_tokenizer': fr_tokenizer,
    'src_pad_idx': train_dataset.en_vocab['<pad>'],
    'tgt_pad_idx': train_dataset.fr_vocab['<pad>'],
    'seed': 0,
    'log_every': 50,  # Number of batches between each wandb logs
}

torch.manual_seed(config['seed'])

config['train_loader'] = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    collate_fn=lambda batch: generate_batch(batch, config['src_pad_idx'], config['tgt_pad_idx'])
)

config['val_loader'] = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    collate_fn=lambda batch: generate_batch(batch, config['src_pad_idx'], config['tgt_pad_idx'])
)

model = TranslationRNN(
    config['n_tokens_src'],
    config['n_tokens_tgt'],
    config['dim_embedding'],
    config['dim_hidden'],
    config['n_layers'],
    config['dropout'],
    config['src_pad_idx'],
    config['tgt_pad_idx'],
    config['model_type']
)

"""
model = TranslationTransformer(
    config['n_tokens_src'],
    config['n_tokens_tgt'],
    config['n_heads'],
    config['dim_embedding'],
    config['dim_hidden'],
    config['n_layers'],
    config['dropout'],
    config['src_pad_idx'],
    config['tgt_pad_idx'],
)
"""

config['optimizer'] = optim.Adam(
    model.parameters(),
    lr=config['lr'],
    betas=config['betas'],
)

weight_classes = torch.ones(config['n_tokens_tgt'], dtype=torch.float)
weight_classes[config['tgt_vocab']['<unk>']] = 0.1  # Lower the importance of that class
config['loss'] = nn.CrossEntropyLoss(
    weight=weight_classes,
    ignore_index=config['tgt_pad_idx'],  # We do not have to learn those
)

summary(
    model,
    input_size=[
        (config['batch_size'], config['max_sequence_length']),
        (config['batch_size'], config['max_sequence_length'])
    ],
    dtypes=[torch.long, torch.long],
    depth=3,
)

Layer (type:depth-idx)                   Output Shape              Param #
TranslationRNN                           [128, 60, 13389]          --
├─Embedding: 1-1                         [128, 60, 196]            1,802,416
├─RNN: 1-2                               [128, 60, 256]            --
│    └─ModuleList: 2-1                   --                        --
│    │    └─RNNCell: 3-1                 [128, 60, 256]            116,224
│    │    └─RNNCell: 3-2                 [128, 60, 256]            131,584
│    │    └─RNNCell: 3-3                 [128, 60, 256]            131,584
├─LayerNorm: 1-3                         [128, 3, 256]             512
├─Embedding: 1-4                         [128, 60, 196]            2,624,244
├─RNN: 1-5                               [128, 60, 256]            --
│    └─ModuleList: 2-2                   --                        --
│    │    └─RNNCell: 3-4                 [128, 60, 256]            116,224
│    │    └─RNNCell: 3-5                 [128, 60,

In [ ]:
# Make sure config has the vocabularies
# config['src_vocab'] = train_dataset.en_vocab
# config['tgt_vocab'] = train_dataset.fr_vocab
# config['src_tokenizer'] = en_tokenizer
# config['tgt_vocab']['<unk>']  # Should be 0
# print("Vocabularies added to config ✓")

In [ ]:
!wandb offline  # online / offline to activate or deactivate WandB logging

torch.autograd.set_detect_anomaly(True)
with wandb.init(
        config=config,
        project='INF8225 - TP3',  # Title of your project
        group='Transformer - small',  # In what group of runs do you want this run to be in?
        save_code=True,
    ):
    train_model(model, config)

wandb: Updated settings file /content/wandb/settings
W&B offline. Running your script from this directory will only write metadata locally. Use `wandb disabled` to completely turn off W&B.


wandb: Loading settings from /content/wandb/settings
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


Starting training for 20 epochs, using cpu.

Epoch 1
Train -   loss: 2.95     top-1: 0.46    top-5: 0.64    top-10: 0.70
Eval -    loss: 2.82     top-1: 0.47    top-5: 0.65    top-10: 0.71
You're a bad liar.
Tu es une bonne idée.

Epoch 2
Train -   loss: 2.65     top-1: 0.49    top-5: 0.67    top-10: 0.74
Eval -    loss: 2.54     top-1: 0.50    top-5: 0.68    top-10: 0.75
My bicycle's been stolen.
Ma mère est mort.

Epoch 3
Train -   loss: 2.50     top-1: 0.50    top-5: 0.69    top-10: 0.76
Eval -    loss: 2.41     top-1: 0.51    top-5: 0.70    top-10: 0.77
Tom is slowly catching up with the rest of the class.
Tom est aussi jeune que la nuit dernière.

Epoch 4
Train -   loss: 2.38     top-1: 0.51    top-5: 0.71    top-10: 0.77
Eval -    loss: 2.31     top-1: 0.53    top-5: 0.72    top-10: 0.78
Ask only questions that can be answered with yes or no.
Il n'y a rien d'autre que moi.

Epoch 5
Train -   loss: 2.29     top-1: 0.52    top-5: 0.72    top-10: 0.78
Eval -    loss: 2.26     top-1:

Train - loss,█▇▆▆▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▃▂▂▂▂▁▂▂▁▁▂▂▁▁▁▂▁
Train - top-1,▁▃▅▅▅▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇████████████████
Train - top-10,▁▂▄▅▄▅▆▆▆▆▆▆▆▆▆▆▆▇▆▇▇▇▇▇▇▇█▇▇█▇▇▇███████
Train - top-5,▁▄▄▄▅▅▅▅▆▆▇▇▇▇▇▇▇▇█▇▇██▇██▇▇▇███████████
Validation - loss,█▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
Validation - top-1,▁▃▄▅▆▆▆▇▇▇▇▇▇▇██████
Validation - top-10,▁▃▄▅▆▆▇▇▇▇▇█████████
Validation - top-5,▁▃▄▅▆▆▇▇▇▇▇▇▇███████
Train - loss,1.87032
Train - top-1,0.56736
Train - top-10,0.84233


In [ ]:
# Debug: Check what the model is actually outputting
def debug_translation(model, english_text):
    """Debug translation to see what's going wrong."""
    print("\n" + "="*70)
    print("TRANSLATION DEBUG")
    print("="*70)

    # Tokenize input
    src_tokens = ['<bos>'] + config['src_tokenizer'](english_text) + ['<eos>']
    print(f"\n1. Input tokens: {src_tokens}")

    # Convert to indices
    src_indices = config['src_vocab'](src_tokens)
    print(f"2. Input indices: {src_indices}")

    # Convert back to check vocab works
    src_back = [config['src_vocab'].lookup_token(idx) for idx in src_indices]
    print(f"3. Converting back: {src_back}")

    # Run model (simple greedy decode)
    model.eval()
    with torch.no_grad():
        src = torch.LongTensor(src_indices).unsqueeze(0).to(config['device'])

        # Start with <bos>
        tgt_indices = [config['tgt_vocab']['<bos>']]
        print(f"\n4. Starting target indices: {tgt_indices}")

        for _ in range(20):  # Generate up to 20 tokens
            tgt = torch.LongTensor(tgt_indices).unsqueeze(0).to(config['device'])

            # Forward pass
            output = model(src, tgt)

            # Get next token prediction
            next_token_logits = output[0, -1, :]
            next_token_idx = next_token_logits.argmax().item()

            print(f"   Step {len(tgt_indices)}: predicted index = {next_token_idx}, token = '{config['tgt_vocab'].lookup_token(next_token_idx)}'")

            tgt_indices.append(next_token_idx)

            if next_token_idx == config['tgt_vocab']['<eos>']:
                break

        print(f"\n5. Final predicted indices: {tgt_indices}")

        # Convert to tokens
        predicted_tokens = [config['tgt_vocab'].lookup_token(idx) for idx in tgt_indices]
        print(f"6. Final predicted tokens: {predicted_tokens}")

        # Check if indices are in vocabulary range
        vocab_size = len(config['tgt_vocab'])
        print(f"\n7. Vocabulary size: {vocab_size}")
        print(f"8. Max predicted index: {max(tgt_indices)}")

        if max(tgt_indices) >= vocab_size:
            print("   ⚠️  ERROR: Model is predicting indices OUTSIDE vocabulary range!")
            print(f"   This means model output dimension doesn't match vocabulary size")

        return predicted_tokens

# Test it
test_sentence = train_dataset.dataset[0][0]  # First training example
debug_translation(model, test_sentence)


TRANSLATION DEBUG

1. Input tokens: ['<bos>', 'What', 'an', 'opportunity', '!', '<eos>']
2. Input indices: [2, 4, 5, 6, 7, 3]
3. Converting back: ['<bos>', 'What', 'an', 'opportunity', '!', '<eos>']

4. Starting target indices: [2]
   Step 1: predicted index = 4, token = 'Quelle'
   Step 2: predicted index = 513, token = 'expérience'
   Step 3: predicted index = 6, token = '!'
   Step 4: predicted index = 3, token = '<eos>'

5. Final predicted indices: [2, 4, 513, 6, 3]
6. Final predicted tokens: ['<bos>', 'Quelle', 'expérience', '!', '<eos>']

7. Vocabulary size: 13389
8. Max predicted index: 513


['<bos>', 'Quelle', 'expérience', '!', '<eos>']

In [ ]:
# Check if model output size matches vocabulary size
print(f"Model output size (n_tokens_tgt): {config['n_tokens_tgt']}")
print(f"French vocabulary size: {len(train_dataset.fr_vocab)}")

if config['n_tokens_tgt'] != len(train_dataset.fr_vocab):
    print("❌ MISMATCH! Model was created with wrong vocabulary size!")
    print("You need to recreate the model with correct n_tokens_tgt")

Model output size (n_tokens_tgt): 13389
French vocabulary size: 13389


In [ ]:
# Verify vocabularies are populated
print(f"\nVocabulary check:")
print(f"English vocab size: {len(train_dataset.en_vocab)}")
print(f"French vocab size: {len(train_dataset.fr_vocab)}")
print(f"First 10 French tokens: {[train_dataset.fr_vocab.lookup_token(i) for i in range(10)]}")

# Check special tokens
for special in ['<unk>', '<pad>', '<bos>', '<eos>']:
    idx = train_dataset.fr_vocab[special]
    print(f"{special} -> {idx}")


Vocabulary check:
English vocab size: 9196
French vocab size: 13389
First 10 French tokens: ['<unk>', '<pad>', '<bos>', '<eos>', 'Quelle', 'occasion', '!', 'Nos', 'ventes', '.']
<unk> -> 0
<pad> -> 1
<bos> -> 2
<eos> -> 3


In [ ]:
sentence = "You can try your work here"
preds = beam_search(
    model,
    sentence,
    config['src_vocab'],
    config['tgt_vocab'],
    config['src_tokenizer'],
    config['device'],
    beam_width=10,
    max_target=100,
    max_sentence_length=config['max_sequence_length']
)[:5]

for i, (translation, likelihood) in enumerate(preds):
    print(f'{i}. ({likelihood*100:.5f}%) \t {translation}')

0. (0.52672%) 	 Tu peux utiliser cette chaise.
1. (0.31951%) 	 Tu peux utiliser cette chemise.
2. (0.31416%) 	 Tu peux répondre à cette question.
3. (0.29808%) 	 Vous pouvez répondre à cette question.
4. (0.29471%) 	 Tu peux faire ça pour toi.


In [ ]:
sentence = "You can try your work here"
preds = greedy_search(model,
              sentence,
              config['src_vocab'],
              config['tgt_vocab'],
              config['src_tokenizer'],
              config['device'],
              config['max_sequence_length']
              )[:5]

for i, (translation, likelihood) in enumerate(preds):
    print(f'{i}. ({likelihood*100:.5f}%) \t {translation}')

0. (0.55486%) 	 <unk> <unk> <unk> <unk> <unk> <unk>


In [ ]:
# Test the model on examples from the training set
import random

print("\n" + "="*70)
print("TESTING MODEL ON TRAINING EXAMPLES")
print("="*70)

# Get a few random examples from the training set
num_examples = 5
random_indices = random.sample(range(len(train_dataset)), num_examples)

for i, idx in enumerate(random_indices):
    print(f"\n--- Example {i+1} ---")

    # Get the original English and French sentences
    en_sentence, fr_sentence = train_dataset.dataset[idx]

    print(f"English (input):  {en_sentence}")
    print(f"French (target):  {fr_sentence}")

    # Use the model to translate
    if 'beam_search' in dir():  # If beam_search is defined
        predictions = beam_search(
            model,
            en_sentence,
            config['src_vocab'],
            config['tgt_vocab'],
            config['src_tokenizer'],
            config['device'],
            beam_width=5,
            max_target=100,
            max_sentence_length=config['max_sequence_length']
        )
        # Get the top prediction
        translation, likelihood = predictions[0]
        print(f"French (predicted): {translation}")
        print(f"Confidence: {likelihood*100:.2f}%")
    elif 'greedy_search' in dir():  # If greedy_search is defined
        translation = greedy_search(
            model,
            en_sentence,
            config['src_vocab'],
            config['tgt_vocab'],
            config['src_tokenizer'],
            config['device'],
            config['max_sequence_length']
        )
        print(f"French (predicted): {translation}")
    else:
        print("No search function defined yet")

print("\n" + "="*70)


TESTING MODEL ON TRAINING EXAMPLES

--- Example 1 ---
English (input):  Have you given up already?
French (target):  As-tu déjà abandonné ?
French (predicted): Avez-vous déjà dîné ?
Confidence: 1.68%

--- Example 2 ---
English (input):  We might die.
French (target):  Il se peut que nous mourions.
French (predicted): Nous pourrions commencer.
Confidence: 5.88%

--- Example 3 ---
English (input):  Thanks for your patience.
French (target):  Merci pour votre patience.
French (predicted): Merci pour votre hospitalité.
Confidence: 12.26%

--- Example 4 ---
English (input):  Everyone has a right to live.
French (target):  Tout le monde a le droit de vivre.
French (predicted): Tout le monde est sur le point de commencer.
Confidence: 0.74%

--- Example 5 ---
English (input):  I'm as tall as my father.
French (target):  Je suis aussi grand que mon père.
French (predicted): Je suis aussi occupée que Tom.
Confidence: 4.71%



# Your Question Responses
1. Explain the differences between Vanilla RNN, GRU-RNN, and Transformers.

etc.

## Effect of head number for the transformer

In [ ]:
new_headnum = 14

In [ ]:

model = TranslationTransformer(
    config['n_tokens_src'],
    config['n_tokens_tgt'],
    new_headnum,
    config['dim_embedding'],
    config['dim_hidden'],
    config['n_layers'],
    config['dropout'],
    config['src_pad_idx'],
    config['tgt_pad_idx']
)


config['optimizer'] = optim.Adam(
    model.parameters(),
    lr=config['lr'],
    betas=config['betas'],
    weight_decay=1e-5 # L2 regularization
)

weight_classes = torch.ones(config['n_tokens_tgt'], dtype=torch.float)
weight_classes[config['tgt_vocab']['<unk>']] = 0.1  # Lower the importance of that class
config['loss'] = nn.CrossEntropyLoss(
    weight=weight_classes,
    ignore_index=config['tgt_pad_idx'],  # We do not have to learn those
)

summary(
    model,
    input_size=[
        (config['batch_size'], config['max_sequence_length']),
        (config['batch_size'], config['max_sequence_length'])
    ],
    dtypes=[torch.long, torch.long],
    depth=3,
)


Layer (type:depth-idx)                             Output Shape              Param #
TranslationTransformer                             [128, 60, 13389]          --
├─Embedding: 1-1                                   [128, 60, 196]            1,802,416
├─PositionalEncoding: 1-2                          [128, 60, 196]            --
│    └─Dropout: 2-1                                [128, 60, 196]            --
├─Embedding: 1-3                                   [128, 60, 196]            2,624,244
├─PositionalEncoding: 1-4                          [128, 60, 196]            --
│    └─Dropout: 2-2                                [128, 60, 196]            --
├─Transformer: 1-5                                 [128, 60, 196]            --
│    └─TransformerEncoder: 2-3                     --                        --
│    │    └─ModuleList: 3-1                        --                        766,344
│    └─TransformerDecoder: 2-4                     --                        --
│    │    └─Modu

In [ ]:
!wandb online  # online / offline to activate or deactivate WandB logging

with wandb.init(
        config=config,
        project='INF8225 - TP3',  # Title of your project
        group='GRU - Experiments',  # In what group of runs do you want this run to be in?
        save_code=True,
    ):
    train_model(model, config)

wandb: Loading settings from /content/wandb/settings
wandb: Updated settings file /content/wandb/settings
W&B online. Running your script from this directory will now sync to the cloud.


Starting training for 20 epochs, using cpu.

Epoch 1
Train -   loss: 2.33     top-1: 0.56    top-5: 0.76    top-10: 0.81
Eval -    loss: 2.11     top-1: 0.58    top-5: 0.78    top-10: 0.82
Of course, I'll marry you.
S'il te plaît.

Epoch 2
Train -   loss: 1.87     top-1: 0.61    top-5: 0.82    top-10: 0.86
Eval -    loss: 1.68     top-1: 0.64    top-5: 0.83    top-10: 0.88
Haggis is a traditional Scottish dish.
Le football est une bonne cuisinière.

Epoch 3
Train -   loss: 1.63     top-1: 0.65    top-5: 0.85    top-10: 0.89
Eval -    loss: 1.47     top-1: 0.67    top-5: 0.86    top-10: 0.90
You can set the white of an egg by boiling it.
Tu peux le dire par an.

Epoch 4
Train -   loss: 1.45     top-1: 0.67    top-5: 0.87    top-10: 0.91
Eval -    loss: 1.37     top-1: 0.69    top-5: 0.87    top-10: 0.91
My house isn't far from here.
Ma maison n'est pas loin d'ici.

Epoch 5
Train -   loss: 1.34     top-1: 0.69    top-5: 0.88    top-10: 0.92
Eval -    loss: 1.29     top-1: 0.70    top-5: 

Train - loss,█▆▅▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▂▁▁▂▂▂▁▂▁
Train - top-1,▁▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██▇▇▇██▇▇█
Train - top-10,▁▆▅▆▆▆▆▇▇▇▇▇▇▇▇▇████████████████████████
Train - top-5,▁▆▇▇▇▇▇▇█▇▇▇████████████████████████████
Validation - loss,█▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
Validation - top-1,▁▃▄▅▆▆▆▇▇▇▇▇████████
Validation - top-10,▁▄▅▆▆▇▇▇▇███████████
Validation - top-5,▁▄▅▆▆▇▇▇▇▇██████████
Train - loss,0.75912
Train - top-1,0.79271
Train - top-10,0.97154


### Testing sentence

In [ ]:
sentence = "Attention is all you need"
preds = beam_search(
    model,
    sentence,
    config['src_vocab'],
    config['tgt_vocab'],
    config['src_tokenizer'],
    config['device'],
    beam_width=10,
    max_target=100,
    max_sentence_length=config['max_sequence_length']
)[:5]

for i, (translation, likelihood) in enumerate(preds):
    print(f'{i}. ({likelihood*100:.5f}%) \t {translation}')

0. (1.40167%) 	 Tout ce dont tu as besoin de ton avis.
1. (1.05109%) 	 Tout ce dont tu as besoin.
2. (0.77950%) 	 Tout ce dont vous avez besoin.
3. (0.71211%) 	 Tout ce dont vous avez besoin, c'est de votre travail.
4. (0.65893%) 	 C'est tout ce dont tu as besoin.


---
# Not Part of TP3, But A Potential Project Idea: Understanding the Architecture of a Decoder-Only Transformer (GPT style) as well as other alternative architectures.

In a project group of 3-4 create a high level plan for a Decoder-Only model for how you would need to modify this code to implement a Decoder-Only Transformer. Key components of the implementation should be split up and each member of the group should present the pseudo-code (or actual code) for one component of the full model to one another, and in a report. Then create the working model and perform experiments comparing it with your TP3 encoder-decoder model.

For more details on the Decoder-Only Transformer see [this blog post](https://medium.com/international-school-of-ai-data-science/building-custom-gpt-with-pytorch-59e5ba8102d4). The [first "GPT" paper](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf), and the paper cited by this GPT-1 paper for the Decoder Only architecture used for GPT, [i.e. this paper](https://arxiv.org/abs/1801.10198). The following Github project might serve as another resource to help you formulate LLM oriented ideas for your project.
https://github.com/jammastergirish/BuildAnLLM
